# FD004 Paper 2 — Degradation-Aware Robust Prognostics Under Distribution Shift

## Purpose

This notebook starts **Paper 2** from the failure mechanism identified and frozen in Paper 1.

Paper 1 showed that:

- Asymmetric CQR is efficient under standard/IID evaluation but loses coverage under both operating-condition shifts.
- Balanced Engine CQR is substantially more stable, maintaining about 90% global coverage across IID, DS-A, and DS-B, but with wider intervals.
- Remaining engine-level reliability failures strongly track **point-model error**, rather than calibration alone.
- The dominant weakness under both distribution shifts is **mid-life degradation**.
- A persistent subgroup of nine engines fails under both DS-A and DS-B.

Therefore, Paper 2 does **not** begin with another calibration variant. It begins by improving the **point predictor**, with special attention to degradation state and mid-life transition dynamics.

---

## Frozen Paper 1 evidence motivating Paper 2

| Evidence | Paper 1 result |
|---|---:|
| DS-A point RMSE | 24.869 |
| DS-B point RMSE | 23.999 |
| DS-A RMSE vs Balanced coverage | -0.953 |
| DS-B RMSE vs Balanced coverage | -0.916 |
| DS-A mid-life Balanced coverage | 70.88% |
| DS-B mid-life Balanced coverage | 75.16% |
| DS-A early-life Balanced coverage | 96.09% |
| DS-B early-life Balanced coverage | 94.72% |
| DS-A late-life Balanced coverage | 90.52% |
| DS-B late-life Balanced coverage | 93.38% |

Persistent shift-sensitive engines:

**17, 105, 120, 156, 181, 184, 194, 240, 245**

### Paper 2 research question

> Can explicit degradation-state information reduce mid-life RUL prediction error and improve robustness under operating-condition distribution shift relative to the frozen Paper 1 regime-aware point-model baseline?

### Primary hypothesis

**H1:** Explicit degradation-state information will reduce mid-life RUL prediction error and improve robustness under operating-condition distribution shift relative to the frozen Paper 1 regime-aware baseline.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Paper 2 experimental roadmap — consolidated after P2-E8 validation

Paper 2 was executed in two phases.

### Phase A — IID/development mechanism search

| Experiment | Model / test | Status | Main result |
|---|---|---|---|
| **P2-E1** | Frozen Paper 1 point-model baseline | **PASS / control** | Global RMSE 18.250; mid-life RMSE 27.739 |
| **P2-E2** | Oracle Early/Mid/Late stage input | **PASS as diagnostic only** | Global RMSE 10.857; mid-life RMSE 11.181 |
| **P2-E3** | Predicted discrete stage probabilities | **FAIL** | Global RMSE 20.100; mid-life RMSE 28.744 |
| **P2-E4** | Continuous degradation auxiliary target | **FAIL** | Global RMSE 19.031; mid-life RMSE 28.267 |
| **P2-E5** | Window-level sensor trend / slope augmentation | **PARTIAL** | Mid-life improves to 26.092, but global RMSE worsens to 19.860 |
| **P2-E6** | Trend-based residual correction | **PARTIAL / not promoted** | Mid-life 26.659, global 18.332 |
| **P2-E6b** | OOF residual correction | **FAIL / not promoted** | Global 18.229; mid-life 27.726; residual-correction correlation 0.044 |
| **P2-E7** | Cycle-to-cycle first-difference augmentation | **FAIL** | Global 19.876; mid-life 31.118 |
| **P2-E8** | Controlled 60-cycle temporal context | **PASS / promoted** | Matched global RMSE 19.116 → **17.432**; matched mid-life 27.704 → **23.788** |

### Phase B — frozen validation gates

| Gate | Status | Main result |
|---|---|---|
| **Shift-V1 — DS-A** | **PASS** | Dedicated P2-E8 point RMSE **16.942** vs Paper 1 **24.869** |
| **Shift-V2 — DS-B** | **PASS** | Dedicated P2-E8 point RMSE **17.130** vs Paper 1 **23.999** |
| **Shift-V3 — persistent nine-engine subgroup** | **PARTIAL PASS / diagnostic limitation remains** | Persistent mid-life RMSE remains ≈32 cycles under both shifts |
| **UQ-V1 — DS-A** | **PARTIAL PASS** | Balanced CQR coverage **85.054%**; mid-life **81.558%** |
| **UQ-V1 — DS-B** | **PASS / near target globally** | Balanced CQR coverage **89.205%**; mid-life **86.349%** |

### Critical experimental rule

The DS-A/DS-B test outcomes and conformal results were kept outside the architecture-development loop. P2-E8 was selected using IID/development evidence first and then frozen for the shift tests.

### Consolidated scientific interpretation

The experiments narrow the original hypothesis. Perfect degradation-stage information is highly informative, but the tested leakage-safe handcrafted stage/trend/delta representations could not recover that oracle benefit. The successful deployable mechanism is **longer temporal context**: extending the observation history from 30 to 60 cycles improves both global and mid-life point prediction and transfers strongly to two distinct operating-condition shifts.

The uncertainty results are more nuanced. **Balanced Engine CQR remains preferable to Asymmetric CQR**, and mid-life Balanced coverage improves substantially relative to Paper 1 under both shifts. However, global Balanced coverage is slightly lower than the Paper 1 reference and the persistent nine-engine subgroup remains undercovered. Paper 2 therefore resolves much of the broad point-prediction failure while leaving a smaller engine-specific reliability problem as the main residual limitation.


## Current status at a glance

The Paper 2 model-development and primary validation sequence is complete through **P2-E8, Shift-V1, Shift-V2, Shift-V3, and UQ-V1**.

**IID matched-endpoint result:** on 8,862 identical validation endpoints, P2-E8 reduces global RMSE from **19.1163 to 17.4316** (**8.8% improvement**) and mid-life RMSE from **27.7041 to 23.7884** (**14.1% improvement**).

**Distribution-shift point results:**
- DS-A: Paper 1 **24.869** → P2-E8 **16.942** RMSE (**31.9% improvement**).
- DS-B: Paper 1 **23.999** → P2-E8 **17.130** RMSE (**28.6% improvement**).

**Persistent-engine diagnostic:** the nine historically difficult engines still have much larger errors than other engines. Their mid-life RMSE is **31.952** under DS-A and **32.350** under DS-B, with positive bias of about **+15.8 cycles** in both shifts.

**Uncertainty validation:** Balanced Engine CQR remains the preferred method. It reaches **85.054%** global coverage on DS-A and **89.205%** on DS-B, with mid-life coverage **81.558%** and **86.349%**, respectively. The persistent subgroup remains the principal coverage weakness.

**Current decision:** freeze **P2-E8 60-cycle context** as the Paper 2 point-model candidate and treat **Balanced Engine CQR** as the preferred uncertainty method. The remaining work is consolidation, reproducibility/freeze packaging, optional multi-seed confirmation, figures/tables, and manuscript writing—not another architecture-search loop.


## Frozen design choices carried forward from Paper 1

These choices are controls and should not be changed casually during Paper 2 model comparison:

- Dataset: NASA C-MAPSS FD004
- RUL clipping: **125 cycles**
- Window length: **30 cycles**
- Selected sensors: **14**
- Operating regimes: **6**
- Regime representation: **one-hot**
- Final point-model input: **14 sensor channels + 6 regime channels = 20 channels**
- Projection before LSTM: **20D → 18D**
- Core point model: **LSTM(64)**
- Seed used for frozen final package: **42**
- Life-stage definition used in the DS-A/DS-B notebooks:
  - **Early:** RUL > 80
  - **Mid:** 40 < RUL ≤ 80
  - **Late:** RUL ≤ 40

Do not use DS-A or DS-B test results for iterative hyperparameter tuning. Use them as final stress tests after selecting the candidate model using development evidence.

# 0. Environment and reproducibility setup

This notebook is written for Google Colab / TensorFlow, matching the existing FD004 research notebooks.

Update the paths below if your Drive layout differs.

In [2]:
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error, mean_absolute_error
import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Concatenate
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

SEED = 42

def set_all_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_all_seeds(SEED)

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
print("Seed:", SEED)

TensorFlow: 2.20.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Seed: 42


In [3]:
# Optional: mount Google Drive when running in Colab.
try:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
except ImportError:
    print("Not running in Colab; Drive mount skipped.")

## 0.1 Configuration

The baseline tensor file is the frozen artifact used in the Paper 1 point-model development notebook.

The expected tensor convention is:

- first 14 channels: selected scaled sensors
- channel 15: integer operating-regime label
- 30 timesteps per sample

Paper 2 converts that regime label to six one-hot channels, yielding the frozen 20-channel representation.

In [4]:
WINDOW_SIZE = 30
RUL_CLIP = 125
N_REGIMES = 6
N_SENSORS = 14
PROJECTION_DIM = 18

SELECTED_SENSORS = [
    "s_2", "s_3", "s_4", "s_7", "s_8", "s_9",
    "s_11", "s_12", "s_13", "s_14",
    "s_15", "s_17", "s_20", "s_21"
]

PERSISTENT_SHIFT_ENGINES = [17, 105, 120, 156, 181, 184, 194, 240, 245]

DATA_PATH = Path("/content/drive/MyDrive/FD004_Research/artifacts/FD004_baseline_tensors.npz")
OUTPUT_DIR = Path("/content/drive/MyDrive/FD004_Paper2_Results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Data path:", DATA_PATH)
print("Output directory:", OUTPUT_DIR)

Data path: /content/drive/MyDrive/FD004_Research/artifacts/FD004_baseline_tensors.npz
Output directory: /content/drive/MyDrive/FD004_Paper2_Results


# 1. P2-E1 — Reproduce the frozen Paper 1 point-model baseline

## Objective

Re-establish the frozen regime-aware point-model control before introducing any degradation-aware information.

The control architecture is the parameter-matched one-hot regime model:

**20 input channels → Dense(18, no bias) → LSTM(64) → Dense(1)**

The purpose of P2-E1 is not to improve performance. It is to create the comparison point for every Paper 2 model.

In [ ]:
from pathlib import Path
import numpy as np
import tensorflow as tf


def load_frozen_baseline_tensors(path=DATA_PATH):
    if not Path(path).exists():
        raise FileNotFoundError(
            f"Baseline tensor file not found: {path}\n"
            "Update DATA_PATH to the frozen Paper 1 FD004_baseline_tensors.npz artifact."
        )

    with np.load(path) as d:
        X_train_raw = d["X_train"]
        y_train = d["y_train"].astype(np.float32)
        X_val_raw = d["X_val"]
        y_val = d["y_val"].astype(np.float32)

    return X_train_raw, y_train, X_val_raw, y_val


def make_onehot_inputs(X_raw):
    sensors = X_raw[:, :, :N_SENSORS].astype(np.float32)
    regime = X_raw[:, :, N_SENSORS].astype(int)

    if regime.min() < 0 or regime.max() >= N_REGIMES:
        raise ValueError(
            f"Expected regime labels in [0, {N_REGIMES-1}], "
            f"found range [{regime.min()}, {regime.max()}]."
        )

    regime_oh = tf.keras.utils.to_categorical(
        regime,
        num_classes=N_REGIMES
    ).astype(np.float32)

    X = np.concatenate([sensors, regime_oh], axis=-1)
    assert X.shape[1:] == (WINDOW_SIZE, N_SENSORS + N_REGIMES)
    return X


# Uncomment after DATA_PATH is correct.
X_train_raw, y_train, X_val_raw, y_val = load_frozen_baseline_tensors()
X_train = make_onehot_inputs(X_train_raw)
X_val = make_onehot_inputs(X_val_raw)
print("Train:", X_train.shape, y_train.shape)
print("Val  :", X_val.shape, y_val.shape)

Train: (43696, 30, 20) (43696,)
Val  : (10332, 30, 20) (10332,)


In [ ]:
def build_frozen_point_baseline():
    inp = Input(
        shape=(WINDOW_SIZE, N_SENSORS + N_REGIMES),
        name="paper1_onehot_input"
    )

    x = Dense(
        PROJECTION_DIM,
        use_bias=False,
        name="linear_projection_18d"
    )(inp)

    x = LSTM(
        64,
        name="lstm_core"
    )(x)

    out = Dense(
        1,
        name="rul_head"
    )(x)

    model = Model(
        inp,
        out,
        name="P2_E1_Frozen_Paper1_Point_Baseline"
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="mse",
        metrics=["mae"]
    )
    return model


baseline_model = build_frozen_point_baseline()
baseline_model.summary()

Model: "P2_E1_Frozen_Paper1_Point_Baseline"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ paper1_onehot_input             │ (None, 30, 20)         │             0 │
│ (InputLayer)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ linear_projection_18d (Dense)   │ (None, 30, 18)         │           360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_core (LSTM)                │ (None, 64)             │        21,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rul_head (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,673 (84.66 KB)

 Trainable params: 21,673 (84.66 KB)

 Non-trainable params: 0 (0.00 B)

## 1.1 Training protocol

For exact reproduction, use the same early-stopping and training protocol as the final controlled Paper 1 point-model comparison.

If your archived Paper 1 model weights are available, loading those weights is preferable for a strict frozen control. Otherwise, retrain with the frozen seed and protocol and record the resulting variation.

In [ ]:
EPOCHS = 50
BATCH_SIZE = 64
PATIENCE = 10

def fit_model(model, X_train, y_train, X_val, y_val, name):
    set_all_seeds(SEED)

    ckpt_path = OUTPUT_DIR / f"{name}.keras"

    callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=PATIENCE,
            restore_best_weights=True,
            verbose=1
        ),
        ModelCheckpoint(
            ckpt_path,
            monitor="val_loss",
            save_best_only=True,
            verbose=0
        )
    ]

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=1
    )

    return history, ckpt_path


# Example:
history_baseline, baseline_ckpt = fit_model(
     baseline_model, X_train, y_train, X_val, y_val, "P2_E1_baseline"
 )

Epoch 1/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - loss: 5315.4385 - mae: 63.6984 - val_loss: 3178.8960 - val_mae: 49.9659
Epoch 2/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 2433.0603 - mae: 44.5897 - val_loss: 1913.8784 - val_mae: 39.9511
Epoch 3/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - loss: 1794.0518 - mae: 38.4752 - val_loss: 1737.9551 - val_mae: 37.2768
Epoch 4/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 1720.3809 - mae: 36.9346 - val_loss: 1734.0507 - val_mae: 36.7534
Epoch 5/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 1500.5364 - mae: 34.0805 - val_loss: 757.8726 - val_mae: 24.1715
Epoch 6/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - loss: 500.8690 - mae: 18.6470 - val_loss: 453.6691 - val_mae: 16.9245
Epoch 7/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 343.0623 - mae: 13.8824 - val_loss: 405.5161 - val_mae: 15.1421
Epoch 8/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 297.8141 - mae: 12.3038 - val_loss: 382.6604 - val_mae: 14.2329

# 2. Evaluation utilities

Paper 2 must not rely only on global RMSE.

Every candidate should be evaluated by:

- overall RMSE / MAE / bias
- early-life RMSE / MAE / bias
- mid-life RMSE / MAE / bias
- late-life RMSE / MAE / bias
- engine-level error when engine IDs are available
- regime-level error
- persistent shift-sensitive subgroup error under DS-A/DS-B

The Paper 1 life-stage boundaries are preserved exactly.

In [ ]:
def assign_life_stage(rul):
    rul = np.asarray(rul)
    return np.where(
        rul > 80, "Early",
        np.where(rul > 40, "Mid", "Late")
    )


def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    err = y_pred - y_true

    return {
        "N": len(y_true),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "Bias_pred_minus_true": float(np.mean(err))
    }


def evaluate_by_life_stage(y_true, y_pred):
    stages = assign_life_stage(y_true)
    rows = []

    for stage in ["Early", "Mid", "Late"]:
        mask = stages == stage
        if not np.any(mask):
            continue

        row = {"Life_Stage": stage}
        row.update(regression_metrics(y_true[mask], y_pred[mask]))
        rows.append(row)

    return pd.DataFrame(rows)


def evaluate_by_group(y_true, y_pred, group_ids, group_name="Group"):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    group_ids = np.asarray(group_ids)

    rows = []
    for g in np.unique(group_ids):
        mask = group_ids == g
        row = {group_name: g}
        row.update(regression_metrics(y_true[mask], y_pred[mask]))
        rows.append(row)

    return pd.DataFrame(rows)


def evaluate_model(model, X, y, label="Model"):
    pred = model.predict(X, verbose=0).reshape(-1)

    global_row = {"Model": label}
    global_row.update(regression_metrics(y, pred))
    global_df = pd.DataFrame([global_row])

    stage_df = evaluate_by_life_stage(y, pred)
    stage_df.insert(0, "Model", label)

    return pred, global_df, stage_df

## 2.1 P2-E1 PASS/FAIL gate

Before moving to degradation-aware models:

1. Verify that input construction is identical to the frozen Paper 1 one-hot representation.
2. Confirm there is no accidental use of DS-A or DS-B test information.
3. Compare baseline RMSE/MAE to the archived Paper 1 baseline result.
4. Save the baseline model, predictions, configuration, and metrics.
5. Record any reproducibility difference explicitly.

**Do not silently accept a materially different baseline.**

In [ ]:
def save_experiment_summary(name, global_df, stage_df, extra_config=None):
    exp_dir = OUTPUT_DIR / name
    exp_dir.mkdir(parents=True, exist_ok=True)

    global_df.to_csv(exp_dir / "global_metrics.csv", index=False)
    stage_df.to_csv(exp_dir / "life_stage_metrics.csv", index=False)

    config = {
        "experiment": name,
        "seed": SEED,
        "window_size": WINDOW_SIZE,
        "rul_clip": RUL_CLIP,
        "n_sensors": N_SENSORS,
        "n_regimes": N_REGIMES,
        "projection_dim": PROJECTION_DIM,
        "life_stage_rule": {
            "Early": "RUL > 80",
            "Mid": "40 < RUL <= 80",
            "Late": "RUL <= 40"
        }
    }

    if extra_config:
        config.update(extra_config)

    with open(exp_dir / "config.json", "w") as f:
        json.dump(config, f, indent=2)

    print("Saved:", exp_dir)

In [ ]:
# ============================================================
# P2-E1 — RUN EVALUATION + DISPLAY RESULTS
# ============================================================

# Evaluate the trained frozen Paper 1 baseline
pred_baseline, global_baseline, stage_baseline = evaluate_model(
    baseline_model,
    X_val,
    y_val,
    label="P2-E1 Frozen Paper1 Baseline"
)

print("\n" + "="*60)
print("P2-E1 GLOBAL VALIDATION PERFORMANCE")
print("="*60)
display(global_baseline)

print("\n" + "="*60)
print("P2-E1 PERFORMANCE BY LIFE STAGE")
print("="*60)
display(stage_baseline)

# Save results
save_experiment_summary(
    name="P2_E1_Frozen_Baseline",
    global_df=global_baseline,
    stage_df=stage_baseline,
    extra_config={
        "training_stopped_epoch": len(history_baseline.history["loss"])
    }
)

print("\nP2-E1 evaluation complete.")


P2-E1 GLOBAL VALIDATION PERFORMANCE


,Model,N,RMSE,MAE,Bias_pred_minus_true
0,P2-E1 Frozen Paper1 Baseline,10332,18.250101,12.66309,-0.432407



P2-E1 PERFORMANCE BY LIFE STAGE


,Model,Life_Stage,N,RMSE,MAE,Bias_pred_minus_true
0,P2-E1 Frozen Paper1 Baseline,Early,6363,16.331137,11.446881,-5.975741
1,P2-E1 Frozen Paper1 Baseline,Mid,1960,27.739052,21.983496,12.866315
2,P2-E1 Frozen Paper1 Baseline,Late,2009,10.839741,7.422046,4.150345


Saved: /content/drive/MyDrive/FD004_Paper2_Results/P2_E1_Frozen_Baseline

P2-E1 evaluation complete.


# 3. P2-E2 — Discrete degradation-stage-aware LSTM

## Idea

Add a degradation-stage representation while keeping the frozen Paper 1 sensor/regime pathway as intact as possible.

To avoid target leakage, the stage feature used for a real prognostic model **must not be computed from true RUL at inference time**.

Therefore this notebook separates two concepts:

1. **Diagnostic oracle-stage experiment** — true RUL stage may be used only as an upper-bound diagnostic to test whether stage information is potentially useful.
2. **Deployable stage-aware experiment** — stage must be estimated from information available at prediction time, such as sensor trajectory / learned health-state features.

The oracle-stage experiment is not a publishable deployment model by itself. It is a mechanistic diagnostic.

In [ ]:
def stage_to_onehot_from_true_rul(y):
    # DIAGNOSTIC ONLY — uses ground-truth RUL and therefore must not be used
    # as an inference-time feature in the final Paper 2 model.
    stage = assign_life_stage(y)
    mapping = {"Early": 0, "Mid": 1, "Late": 2}
    idx = np.array([mapping[s] for s in stage])
    return tf.keras.utils.to_categorical(idx, num_classes=3).astype(np.float32)


def append_constant_sequence_feature(X, feature_vector):
    # feature_vector shape: (N, K)
    feature_vector = np.asarray(feature_vector, dtype=np.float32)
    repeated = np.repeat(feature_vector[:, None, :], X.shape[1], axis=1)
    return np.concatenate([X, repeated], axis=-1)


def build_oracle_stage_model(input_dim=23):
    inp = Input(shape=(WINDOW_SIZE, input_dim), name="stage_augmented_input")
    x = Dense(PROJECTION_DIM, use_bias=False, name="linear_projection_18d")(inp)
    x = LSTM(64, name="lstm_core")(x)
    out = Dense(1, name="rul_head")(x)

    model = Model(inp, out, name="P2_E2_Oracle_Stage_Diagnostic")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="mse",
        metrics=["mae"]
    )
    return model


# Example diagnostic construction:
stage_train_oh = stage_to_onehot_from_true_rul(y_train)
stage_val_oh = stage_to_onehot_from_true_rul(y_val)
X_train_oracle_stage = append_constant_sequence_feature(X_train, stage_train_oh)
X_val_oracle_stage = append_constant_sequence_feature(X_val, stage_val_oh)

P2-E1 — PASS

Global RMSE: 18.2501
Global MAE: 12.6631
Global bias: -0.4324

Early RMSE: 16.3311
Mid RMSE:   27.7391
Late RMSE:  10.8397

Main finding:
The frozen Paper 1 baseline was successfully reproduced.
Mid-life is the dominant error region and shows strong positive bias,
supporting progression to degradation-aware modeling.

## P2-E2 decision rule

The important question is not “does the oracle stage model have a lower global RMSE?”

The important question is:

> Does stage information materially reduce **mid-life RMSE**, while avoiding a substantial penalty in early and late life?

If the answer is no, stop pursuing explicit stage conditioning and reconsider the Paper 2 mechanism.

If the answer is yes, proceed to a deployable degradation-state estimator.

In [ ]:
# ============================================================
# P2-E2 — ORACLE STAGE DIAGNOSTIC
# Diagnostic only: true RUL stage is used as input.
# This tests whether degradation-stage information is useful.
# ============================================================

# Build oracle stage features
stage_train_oh = stage_to_onehot_from_true_rul(y_train)
stage_val_oh = stage_to_onehot_from_true_rul(y_val)

X_train_oracle_stage = append_constant_sequence_feature(
    X_train,
    stage_train_oh
)

X_val_oracle_stage = append_constant_sequence_feature(
    X_val,
    stage_val_oh
)

print("Baseline input shape:", X_train.shape)
print("Oracle-stage input shape:", X_train_oracle_stage.shape)

# Build model
oracle_stage_model = build_oracle_stage_model(
    input_dim=X_train_oracle_stage.shape[-1]
)

oracle_stage_model.summary()

# Train
history_oracle, oracle_ckpt = fit_model(
    oracle_stage_model,
    X_train_oracle_stage,
    y_train,
    X_val_oracle_stage,
    y_val,
    "P2_E2_Oracle_Stage"
)

print("Training stopped after:",
      len(history_oracle.history["loss"]),
      "epochs")

Baseline input shape: (43696, 30, 20)
Oracle-stage input shape: (43696, 30, 23)


Model: "P2_E2_Oracle_Stage_Diagnostic"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ stage_augmented_input           │ (None, 30, 23)         │             0 │
│ (InputLayer)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ linear_projection_18d (Dense)   │ (None, 30, 18)         │           414 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_core (LSTM)                │ (None, 64)             │        21,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rul_head (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,727 (84.87 KB)

 Trainable params: 21,727 (84.87 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - loss: 5361.7603 - mae: 63.9896 - val_loss: 3213.1938 - val_mae: 50.2012
Epoch 2/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 2084.3120 - mae: 39.1382 - val_loss: 1081.0181 - val_mae: 27.7431
Epoch 3/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 646.9161 - mae: 21.4447 - val_loss: 336.1864 - val_mae: 16.1851
Epoch 4/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - loss: 216.3587 - mae: 12.9514 - val_loss: 152.1196 - val_mae: 10.7935
Epoch 5/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 136.8969 - mae: 9.8484 - val_loss: 135.6859 - val_mae: 9.4221
Epoch 6/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 128.9552 - mae: 9.0489 - val_loss: 132.6205 - val_mae: 9.1746
Epoch 7/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - loss: 128.0051 - mae: 9.1591 - val_loss: 131.7956 - val_mae: 9.0136
Epoch 8/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 122.8533 - mae: 8.6440 - val_loss: 126.8405 - val_mae: 8.6188
Epoch 9/50
6

In [ ]:
# ============================================================
# P2-E2 — EVALUATION
# ============================================================

pred_oracle, global_oracle, stage_oracle = evaluate_model(
    oracle_stage_model,
    X_val_oracle_stage,
    y_val,
    label="P2-E2 Oracle Stage"
)

print("\n" + "="*60)
print("P2-E2 GLOBAL VALIDATION PERFORMANCE")
print("="*60)
display(global_oracle)

print("\n" + "="*60)
print("P2-E2 PERFORMANCE BY LIFE STAGE")
print("="*60)
display(stage_oracle)


P2-E2 GLOBAL VALIDATION PERFORMANCE


,Model,N,RMSE,MAE,Bias_pred_minus_true
0,P2-E2 Oracle Stage,10332,10.857481,8.050015,0.205902



P2-E2 PERFORMANCE BY LIFE STAGE


,Model,Life_Stage,N,RMSE,MAE,Bias_pred_minus_true
0,P2-E2 Oracle Stage,Early,6363,12.008667,9.013854,0.097943
1,P2-E2 Oracle Stage,Mid,1960,11.180762,8.954518,1.621357
2,P2-E2 Oracle Stage,Late,2009,5.249957,4.114860,-0.833099


In [ ]:
# ============================================================
# P2-E1 vs P2-E2 COMPARISON
# ============================================================

comparison = pd.concat(
    [stage_baseline, stage_oracle],
    ignore_index=True
)

display(comparison)

pivot = comparison.pivot(
    index="Life_Stage",
    columns="Model",
    values="RMSE"
)

pivot["RMSE_Change"] = (
    pivot["P2-E2 Oracle Stage"] -
    pivot["P2-E1 Frozen Paper1 Baseline"]
)

pivot["Improvement_%"] = (
    (
        pivot["P2-E1 Frozen Paper1 Baseline"] -
        pivot["P2-E2 Oracle Stage"]
    )
    / pivot["P2-E1 Frozen Paper1 Baseline"]
    * 100
)

display(pivot)

,Model,Life_Stage,N,RMSE,MAE,Bias_pred_minus_true
0,P2-E1 Frozen Paper1 Baseline,Early,6363,16.331137,11.446881,-5.975741
1,P2-E1 Frozen Paper1 Baseline,Mid,1960,27.739052,21.983496,12.866315
2,P2-E1 Frozen Paper1 Baseline,Late,2009,10.839741,7.422046,4.150345
3,P2-E2 Oracle Stage,Early,6363,12.008667,9.013854,0.097943
4,P2-E2 Oracle Stage,Mid,1960,11.180762,8.954518,1.621357
5,P2-E2 Oracle Stage,Late,2009,5.249957,4.114860,-0.833099


Model,P2-E1 Frozen Paper1 Baseline,P2-E2 Oracle Stage,RMSE_Change,Improvement_%
Life_Stage,,,,
Early,16.331137,12.008667,-4.322470,26.467659
Late,10.839741,5.249957,-5.589784,51.567507
Mid,27.739052,11.180762,-16.558291,59.693066


# 4. P2-E3 — Leakage-safe predicted degradation-stage representation

## Objective

Test whether the very large oracle-stage benefit from P2-E2 can be recovered using **stage information inferred only from the observed 30-cycle sensor/regime window**.

A three-class classifier predicts probabilities for:

- Early
- Mid
- Late

Those **predicted probabilities**, rather than true RUL-derived labels, are appended to the RUL model input.

### Why this experiment matters

P2-E2 proved that perfect stage information is highly informative, but it is not deployable because true RUL is unavailable at inference time. P2-E3 therefore tests whether the stage can be estimated from observable history without leakage.

### Result

The classifier reached 82.0% overall stage accuracy, but **Mid recall was only 45.36%**. The downstream RUL model became worse than the frozen baseline:

- Global RMSE: **20.100**
- Early RMSE: **18.597**
- Mid RMSE: **28.744**
- Late RMSE: **13.274**

**Decision: FAIL / do not promote.**

The coarse three-class inferred representation does not recover the oracle benefit, especially in the transition region that Paper 2 is trying to repair.

In [ ]:
# P2-E3 scaffold.
# Replace this placeholder with a degradation index that is computable from
# sensor/operating-history information only.

def compute_degradation_index_placeholder(X):
    raise NotImplementedError(
        "Implement a leakage-safe degradation index derived from observed history only."
    )


def build_continuous_index_model(input_dim=21):
    inp = Input(shape=(WINDOW_SIZE, input_dim), name="degradation_index_input")
    x = Dense(PROJECTION_DIM, use_bias=False)(inp)
    x = LSTM(64)(x)
    out = Dense(1)(x)

    model = Model(inp, out, name="P2_E3_Continuous_Degradation_Index")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="mse",
        metrics=["mae"]
    )
    return model

In [ ]:
# ============================================================
# P2-E3 — LEAKAGE-SAFE LEARNED DEGRADATION STATE
# ============================================================

from tensorflow.keras.layers import Input, Dense, LSTM, RepeatVector
from tensorflow.keras.models import Model


def build_degradation_stage_classifier(input_dim=20):
    inp = Input(
        shape=(WINDOW_SIZE, input_dim),
        name="input_window"
    )

    x = Dense(
        PROJECTION_DIM,
        use_bias=False,
        name="projection"
    )(inp)

    x = LSTM(
        64,
        name="stage_encoder"
    )(x)

    out = Dense(
        3,
        activation="softmax",
        name="stage_probabilities"
    )(x)

    model = Model(
        inp,
        out,
        name="P2_E3_Stage_Classifier"
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [ ]:
def stage_labels_from_rul(y):
    y = np.asarray(y).reshape(-1)

    # 0 = Early, 1 = Mid, 2 = Late
    labels = np.where(
        y > 80,
        0,
        np.where(y > 40, 1, 2)
    )

    return labels.astype(np.int32)


stage_train = stage_labels_from_rul(y_train)
stage_val = stage_labels_from_rul(y_val)

print(pd.Series(stage_train).value_counts().sort_index())

0    27496
1     8000
2     8200
Name: count, dtype: int64


In [ ]:
stage_classifier = build_degradation_stage_classifier(
    input_dim=X_train.shape[-1]
)

history_stage_classifier, stage_classifier_ckpt = fit_model(
    stage_classifier,
    X_train,
    stage_train,
    X_val,
    stage_val,
    "P2_E3_Stage_Classifier"
)

Epoch 1/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.7956 - loss: 0.4973 - val_accuracy: 0.7889 - val_loss: 0.5136
Epoch 2/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.8396 - loss: 0.3895 - val_accuracy: 0.8113 - val_loss: 0.4739
Epoch 3/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.8513 - loss: 0.3588 - val_accuracy: 0.8121 - val_loss: 0.4720
Epoch 4/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.8635 - loss: 0.3319 - val_accuracy: 0.8200 - val_loss: 0.4649
Epoch 5/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.8736 - loss: 0.3091 - val_accuracy: 0.8188 - val_loss: 0.4757
Epoch 6/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.8823 - loss: 0.2881 - val_accuracy: 0.8156 - val_loss: 0.5069
Epoch 7/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.8904 - loss: 0.2690 - val_accuracy: 0.8026 - val_loss: 0.5562
Epoch 8/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.8990 - loss: 0.2492 - val_accuracy: 0.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

stage_prob_train = stage_classifier.predict(
    X_train,
    verbose=0
)

stage_prob_val = stage_classifier.predict(
    X_val,
    verbose=0
)

stage_pred_val = np.argmax(
    stage_prob_val,
    axis=1
)

print(classification_report(
    stage_val,
    stage_pred_val,
    target_names=["Early", "Mid", "Late"],
    digits=4
))

print("Confusion matrix:")
print(confusion_matrix(stage_val, stage_pred_val))

              precision    recall  f1-score   support

       Early     0.8921    0.9126    0.9023      6363
         Mid     0.5612    0.4536    0.5017      1960
        Late     0.7932    0.8840    0.8362      2009

    accuracy                         0.8200     10332
   macro avg     0.7489    0.7501    0.7467     10332
weighted avg     0.8101    0.8200    0.8134     10332

Confusion matrix:
[[5807  492   64]
 [ 672  889  399]
 [  30  203 1776]]


In [ ]:
X_train_pred_stage = append_constant_sequence_feature(
    X_train,
    stage_prob_train
)

X_val_pred_stage = append_constant_sequence_feature(
    X_val,
    stage_prob_val
)

print("New train shape:", X_train_pred_stage.shape)
print("New val shape:", X_val_pred_stage.shape)

New train shape: (43696, 30, 23)
New val shape: (10332, 30, 23)


In [ ]:
p2e3_model = build_oracle_stage_model(
    input_dim=X_train_pred_stage.shape[-1]
)

p2e3_model._name = "P2_E3_Predicted_Stage_RUL"

history_p2e3, p2e3_ckpt = fit_model(
    p2e3_model,
    X_train_pred_stage,
    y_train,
    X_val_pred_stage,
    y_val,
    "P2_E3_Predicted_Stage_RUL"
)

Epoch 1/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - loss: 5361.8569 - mae: 63.9897 - val_loss: 3212.9304 - val_mae: 50.1995
Epoch 2/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 2163.8577 - mae: 40.2371 - val_loss: 1218.4929 - val_mae: 29.9391
Epoch 3/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - loss: 757.4227 - mae: 23.3371 - val_loss: 566.9868 - val_mae: 20.6303
Epoch 4/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 363.0129 - mae: 16.0980 - val_loss: 427.7397 - val_mae: 16.8820
Epoch 5/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 272.2179 - mae: 12.9838 - val_loss: 408.0162 - val_mae: 15.0772
Epoch 6/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - loss: 250.8401 - mae: 11.6603 - val_loss: 408.3744 - val_mae: 14.3416
Epoch 7/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 247.2482 - mae: 11.2040 - val_loss: 405.2690 - val_mae: 14.3453
Epoch 8/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 241.8030 - mae: 10.9095 - val_loss: 408.5113 - val_mae: 14.1979
Epoc

In [ ]:
print(

    "P2-E3 training stopped after:",

    len(history_p2e3.history["loss"]),

    "epochs"

)

P2-E3 training stopped after: 19 epochs


In [ ]:
# ============================================================
# P2-E3 — EVALUATION
# ============================================================

pred_p2e3, global_p2e3, stage_p2e3 = evaluate_model(
    p2e3_model,
    X_val_pred_stage,
    y_val,
    label="P2-E3 Predicted Stage"
)

print("\n" + "="*60)
print("P2-E3 GLOBAL VALIDATION PERFORMANCE")
print("="*60)
display(global_p2e3)

print("\n" + "="*60)
print("P2-E3 PERFORMANCE BY LIFE STAGE")
print("="*60)
display(stage_p2e3)


P2-E3 GLOBAL VALIDATION PERFORMANCE


,Model,N,RMSE,MAE,Bias_pred_minus_true
0,P2-E3 Predicted Stage,10332,20.099595,13.957098,-1.793231



P2-E3 PERFORMANCE BY LIFE STAGE


,Model,Life_Stage,N,RMSE,MAE,Bias_pred_minus_true
0,P2-E3 Predicted Stage,Early,6363,18.597106,12.831351,-7.645110
1,P2-E3 Predicted Stage,Mid,1960,28.744333,22.607552,9.848330
2,P2-E3 Predicted Stage,Late,2009,13.273933,9.083141,5.383496


In [ ]:
# ============================================================
# P2-E1 vs P2-E2 vs P2-E3
# ============================================================

all_stage_results = pd.concat(
    [
        stage_baseline,
        stage_oracle,
        stage_p2e3
    ],
    ignore_index=True
)

display(all_stage_results)

comparison = all_stage_results.pivot(
    index="Life_Stage",
    columns="Model",
    values="RMSE"
)

display(comparison)

,Model,Life_Stage,N,RMSE,MAE,Bias_pred_minus_true
0,P2-E1 Frozen Paper1 Baseline,Early,6363,16.331137,11.446881,-5.975741
1,P2-E1 Frozen Paper1 Baseline,Mid,1960,27.739052,21.983496,12.866315
2,P2-E1 Frozen Paper1 Baseline,Late,2009,10.839741,7.422046,4.150345
3,P2-E2 Oracle Stage,Early,6363,12.008667,9.013854,0.097943
4,P2-E2 Oracle Stage,Mid,1960,11.180762,8.954518,1.621357
5,P2-E2 Oracle Stage,Late,2009,5.249957,4.114860,-0.833099
6,P2-E3 Predicted Stage,Early,6363,18.597106,12.831351,-7.645110
7,P2-E3 Predicted Stage,Mid,1960,28.744333,22.607552,9.848330
8,P2-E3 Predicted Stage,Late,2009,13.273933,9.083141,5.383496


Model,P2-E1 Frozen Paper1 Baseline,P2-E2 Oracle Stage,P2-E3 Predicted Stage
Life_Stage,,,
Early,16.331137,12.008667,18.597106
Late,10.839741,5.249957,13.273933
Mid,27.739052,11.180762,28.744333


In [ ]:
# ============================================================
# P2-E3 IMPROVEMENT RELATIVE TO BASELINE
# ============================================================

baseline_rmse = (
    stage_baseline
    .set_index("Life_Stage")["RMSE"]
)

p2e3_rmse = (
    stage_p2e3
    .set_index("Life_Stage")["RMSE"]
)

improvement = pd.DataFrame({
    "P2-E1 Baseline RMSE": baseline_rmse,
    "P2-E3 RMSE": p2e3_rmse
})

improvement["RMSE Change"] = (
    improvement["P2-E3 RMSE"]
    - improvement["P2-E1 Baseline RMSE"]
)

improvement["Improvement %"] = (
    (
        improvement["P2-E1 Baseline RMSE"]
        - improvement["P2-E3 RMSE"]
    )
    / improvement["P2-E1 Baseline RMSE"]
    * 100
)

display(improvement)

,P2-E1 Baseline RMSE,P2-E3 RMSE,RMSE Change,Improvement %
Life_Stage,,,,
Early,16.331137,18.597106,2.265969,-13.875149
Mid,27.739052,28.744333,1.005281,-3.624064
Late,10.839741,13.273933,2.434192,-22.456178


P2-E3 — FAIL / DO NOT PROMOTE

Global RMSE:
P2-E1 = 18.2501
P2-E3 = 20.0996

Mid-life RMSE:
P2-E1 = 27.7391
P2-E3 = 28.7443

Finding:
Oracle degradation-stage information is highly valuable, but a
three-class inferred stage representation does not recover that benefit.

Likely mechanism:
The degradation transition is continuous rather than cleanly separable
into Early/Mid/Late classes. Mid-stage classification recall was only
45.36%, and prediction errors around the stage boundaries propagate
into the RUL predictor.

Decision:
Reject discrete predicted-stage conditioning.
Proceed to a continuous learned degradation representation.

# 5. P2-E4 — Continuous degradation auxiliary supervision

## Objective

Test whether a continuous degradation target can improve the shared temporal representation without using a discrete inferred stage.

The auxiliary degradation target is defined from clipped RUL during training only:

**degradation = 1 - clipped_RUL / 125**

The model uses a shared LSTM representation with two heads:

- primary RUL regression head
- auxiliary continuous degradation head

### Result

The auxiliary task did **not** improve the frozen point model:

- Global RMSE: **19.031**
- Early RMSE: **17.074**
- Mid RMSE: **28.267**
- Late RMSE: **12.640**

**Decision: FAIL / do not promote.**

Because this degradation target is almost a rescaled version of the RUL target, it supplies little independent information about the mid-life transition dynamics.

In [ ]:
# P2-E4 template: multi-task degradation-aware model.
# This is intentionally a scaffold rather than a frozen choice.

def build_multitask_degradation_model(input_dim=20):
    inp = Input(shape=(WINDOW_SIZE, input_dim), name="input_window")
    x = Dense(PROJECTION_DIM, use_bias=False, name="projection")(inp)
    latent = LSTM(64, name="degradation_encoder")(x)

    rul_out = Dense(1, name="rul")(latent)
    stage_out = Dense(3, activation="softmax", name="stage")(latent)

    model = Model(
        inp,
        {"rul": rul_out, "stage": stage_out},
        name="P2_E4_Multitask_Degradation_Model"
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss={
            "rul": "mse",
            "stage": "sparse_categorical_crossentropy"
        },
        loss_weights={
            "rul": 1.0,
            "stage": 0.1
        },
        metrics={
            "rul": ["mae"],
            "stage": ["accuracy"]
        }
    )
    return model

In [ ]:
# ============================================================
# P2-E4 — CONTINUOUS DEGRADATION TARGET
# ============================================================

def degradation_target_from_rul(y, rul_clip=125.0):
    y = np.asarray(y, dtype=np.float32).reshape(-1)

    y_clip = np.clip(y, 0.0, rul_clip)

    degradation = 1.0 - (y_clip / rul_clip)

    return degradation.astype(np.float32)


d_train = degradation_target_from_rul(y_train)
d_val = degradation_target_from_rul(y_val)

print("Degradation target range:")
print("Train:", d_train.min(), d_train.max())
print("Val  :", d_val.min(), d_val.max())

print("\nExamples:")
for rul in [125, 100, 80, 60, 40, 20, 0]:
    d = degradation_target_from_rul([rul])[0]
    print(f"RUL={rul:3d} -> degradation={d:.3f}")

Degradation target range:
Train: 0.0 1.0
Val  : 0.0 1.0

Examples:
RUL=125 -> degradation=0.000
RUL=100 -> degradation=0.200
RUL= 80 -> degradation=0.360
RUL= 60 -> degradation=0.520
RUL= 40 -> degradation=0.680
RUL= 20 -> degradation=0.840
RUL=  0 -> degradation=1.000


In [ ]:
# ============================================================
# P2-E4 — MULTI-TASK CONTINUOUS DEGRADATION MODEL
# ============================================================

from tensorflow.keras.layers import Input, Dense, LSTM
from tensorflow.keras.models import Model


def build_continuous_degradation_multitask(input_dim=20):

    inp = Input(
        shape=(WINDOW_SIZE, input_dim),
        name="input_window"
    )

    x = Dense(
        PROJECTION_DIM,
        use_bias=False,
        name="projection"
    )(inp)

    latent = LSTM(
        64,
        name="degradation_encoder"
    )(x)

    # Primary task
    rul_out = Dense(
        1,
        name="rul"
    )(latent)

    # Auxiliary continuous degradation task
    degradation_out = Dense(
        1,
        activation="sigmoid",
        name="degradation"
    )(latent)

    model = Model(
        inputs=inp,
        outputs={
            "rul": rul_out,
            "degradation": degradation_out
        },
        name="P2_E4_Continuous_Degradation_Multitask"
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),

        loss={
            "rul": "mse",
            "degradation": "mse"
        },

        loss_weights={
            "rul": 1.0,
            "degradation": 0.1
        },

        metrics={
            "rul": ["mae"],
            "degradation": ["mae"]
        }
    )

    return model

In [ ]:
# ============================================================
# TRAIN P2-E4
# ============================================================

set_all_seeds(SEED)

p2e4_model = build_continuous_degradation_multitask(
    input_dim=X_train.shape[-1]
)

p2e4_model.summary()

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_rul_loss",
        mode="min",
        patience=10,
        restore_best_weights=True,
        verbose=1
    )
]

history_p2e4 = p2e4_model.fit(
    X_train,
    {
        "rul": y_train,
        "degradation": d_train
    },

    validation_data=(
        X_val,
        {
            "rul": y_val,
            "degradation": d_val
        }
    ),

    epochs=50,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

print(
    "Training stopped after:",
    len(history_p2e4.history["loss"]),
    "epochs"
)

Model: "P2_E4_Continuous_Degradation_Multitask"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_window        │ (None, 30, 20)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ projection (Dense)  │ (None, 30, 18)    │        360 │ input_window[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ degradation_encoder │ (None, 64)        │     21,248 │ projection[0][0]  │
│ (LSTM)              │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ degradation (Dense) │ (None, 1)         │         65 │ degradation_enco… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rul (Dense)         │ (None, 1)         │         65 │ degradation_enco… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 21,738 (84.91 KB)

 Trainable params: 21,738 (84.91 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - degradation_loss: 0.1111 - degradation_mae: 0.2939 - loss: 5315.4482 - rul_loss: 5314.9067 - rul_mae: 63.6984 - val_degradation_loss: 0.1120 - val_degradation_mae: 0.2933 - val_loss: 3178.9067 - val_rul_loss: 3172.6311 - val_rul_mae: 49.9659
Epoch 2/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - degradation_loss: 0.1103 - degradation_mae: 0.2932 - loss: 2433.0728 - rul_loss: 2432.9429 - rul_mae: 44.5897 - val_degradation_loss: 0.1120 - val_degradation_mae: 0.2933 - val_loss: 1913.8900 - val_rul_loss: 1920.2806 - val_rul_mae: 39.9511
Epoch 3/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - degradation_loss: 0.1103 - degradation_mae: 0.2932 - loss: 1794.0626 - rul_loss: 1794.0214 - rul_mae: 38.4752 - val_degradation_loss: 0.1120 - val_degradation_mae: 0.2933 - val_loss: 1737.9653 - val_rul_loss: 1750.1625 - val_rul_mae: 37.2768
Epoch 4/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - degradation_loss: 0.1103 - degradation_mae: 0.2932 - loss:

In [ ]:
# ============================================================
# P2-E4 — RUL EVALUATION
# ============================================================

outputs = p2e4_model.predict(
    X_val,
    verbose=0
)

pred_p2e4 = outputs["rul"].reshape(-1)
pred_degradation = outputs["degradation"].reshape(-1)

global_p2e4_row = {
    "Model": "P2-E4 Continuous Degradation",
    **regression_metrics(y_val, pred_p2e4)
}

global_p2e4 = pd.DataFrame(
    [global_p2e4_row]
)

stage_p2e4 = evaluate_by_life_stage(
    y_val,
    pred_p2e4
)

stage_p2e4.insert(
    0,
    "Model",
    "P2-E4 Continuous Degradation"
)

print("\nGLOBAL PERFORMANCE")
display(global_p2e4)

print("\nLIFE-STAGE PERFORMANCE")
display(stage_p2e4)


GLOBAL PERFORMANCE


,Model,N,RMSE,MAE,Bias_pred_minus_true
0,P2-E4 Continuous Degradation,10332,19.030789,13.169939,-0.203915



LIFE-STAGE PERFORMANCE


,Model,Life_Stage,N,RMSE,MAE,Bias_pred_minus_true
0,P2-E4 Continuous Degradation,Early,6363,17.073940,11.746227,-6.029100
1,P2-E4 Continuous Degradation,Mid,1960,28.266745,22.369209,12.730289
2,P2-E4 Continuous Degradation,Late,2009,12.639549,8.704293,5.627151


In [ ]:
comparison_p2e4 = pd.concat(
    [
        stage_baseline,
        stage_oracle,
        stage_p2e4
    ],
    ignore_index=True
)

pivot_p2e4 = comparison_p2e4.pivot(
    index="Life_Stage",
    columns="Model",
    values="RMSE"
)

display(pivot_p2e4)

Model,P2-E1 Frozen Paper1 Baseline,P2-E2 Oracle Stage,P2-E4 Continuous Degradation
Life_Stage,,,
Early,16.331137,12.008667,17.073940
Late,10.839741,5.249957,12.639549
Mid,27.739052,11.180762,28.266745


P2-E4 — FAIL / DO NOT PROMOTE

Global RMSE:
P2-E1 = 18.2501
P2-E4 = 19.0308

Mid-life RMSE:
P2-E1 = 27.7391
P2-E4 = 28.2667

Finding:
A continuous auxiliary degradation target derived directly from RUL
does not improve the frozen point model.

Interpretation:
The auxiliary degradation target is largely redundant with the RUL
regression target and does not provide independent information about
mid-life degradation dynamics.

Decision:
Do not continue tuning the auxiliary-loss weight.
Move to explicit temporal degradation-dynamics features.

# 6. Multi-seed robustness gate

Paper 1 already established the importance of repeatability.

A candidate Paper 2 model should not be promoted on the basis of one favorable seed.

Recommended seed set, consistent with prior controlled experiments:

**[42, 123, 456, 789, 1000]**

At minimum, report:

- mean ± standard deviation of global RMSE
- mean ± standard deviation of mid-life RMSE
- early-life RMSE
- late-life RMSE
- MAE
- bias
- parameter count

The primary model-selection criterion should emphasize **mid-life error reduction with stable overall performance**, not only the lowest single-seed global RMSE.

In [ ]:
MULTI_SEEDS = [42, 123, 456, 789, 1000]

def summarize_multiseed(records):
    df = pd.DataFrame(records)

    numeric_cols = [
        c for c in df.columns
        if pd.api.types.is_numeric_dtype(df[c]) and c != "Seed"
    ]

    summary = []
    for col in numeric_cols:
        summary.append({
            "Metric": col,
            "Mean": df[col].mean(),
            "Std": df[col].std(ddof=1)
        })

    return df, pd.DataFrame(summary)

# 6. P2-E5 — Window-level sensor trend / slope augmentation

## Objective

Add leakage-safe temporal trend information computed from the observed 30-cycle window.

For each selected sensor, one linear slope is estimated across the window and repeated across timesteps. The resulting input contains:

- 14 sensor levels
- 14 sensor slopes
- 6 regime one-hot channels

Total: **34 channels**.

### Result

P2-E5 is the first leakage-safe experiment to improve the target mid-life region:

- Global RMSE: **19.860** — worse than baseline
- Early RMSE: **19.613** — worse
- Mid RMSE: **26.092** — **5.9% better than baseline**
- Late RMSE: **12.077** — worse

**Decision: PARTIAL / mechanism supported, model not promoted.**

The slopes contain some useful transition information, but injecting them indiscriminately harms the easier Early and Late regions.

In [ ]:
# ============================================================
# P2-E5 — SENSOR TREND FEATURES
# ============================================================

def compute_sensor_slopes(X_raw):
    """
    Compute one linear slope per sensor for each 30-cycle window.

    X_raw:
        shape (N, 30, 15)
        first 14 channels = sensors
        channel 14 = regime label

    Returns:
        slopes shape (N, 14)
    """

    sensors = X_raw[:, :, :N_SENSORS].astype(np.float32)

    n_samples, timesteps, n_sensors = sensors.shape

    t = np.arange(timesteps, dtype=np.float32)

    t_centered = t - t.mean()

    denominator = np.sum(t_centered ** 2)

    sensor_centered = (
        sensors
        - sensors.mean(axis=1, keepdims=True)
    )

    slopes = np.sum(
        sensor_centered * t_centered[None, :, None],
        axis=1
    ) / denominator

    return slopes.astype(np.float32)

In [ ]:
def make_trend_aware_inputs(X_raw):

    # Original frozen Paper 1 representation
    X_base = make_onehot_inputs(X_raw)

    # One slope value per sensor/window
    slopes = compute_sensor_slopes(X_raw)

    # Repeat slopes across the 30 timesteps
    slopes_seq = np.repeat(
        slopes[:, None, :],
        WINDOW_SIZE,
        axis=1
    )

    X_trend = np.concatenate(
        [
            X_base,
            slopes_seq
        ],
        axis=-1
    )

    return X_trend.astype(np.float32)

In [ ]:
X_train_trend = make_trend_aware_inputs(
    X_train_raw
)

X_val_trend = make_trend_aware_inputs(
    X_val_raw
)

print("Baseline shape:", X_train.shape)
print("Trend-aware shape:", X_train_trend.shape)

Baseline shape: (43696, 30, 20)
Trend-aware shape: (43696, 30, 34)


In [ ]:
def build_trend_aware_model(input_dim=34):

    inp = Input(
        shape=(WINDOW_SIZE, input_dim),
        name="trend_aware_input"
    )

    x = Dense(
        PROJECTION_DIM,
        use_bias=False,
        name="projection"
    )(inp)

    x = LSTM(
        64,
        name="lstm_core"
    )(x)

    out = Dense(
        1,
        name="rul"
    )(x)

    model = Model(
        inp,
        out,
        name="P2_E5_Trend_Aware_LSTM"
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="mse",
        metrics=["mae"]
    )

    return model

In [ ]:
p2e5_model = build_trend_aware_model(
    input_dim=X_train_trend.shape[-1]
)

history_p2e5, p2e5_ckpt = fit_model(
    p2e5_model,
    X_train_trend,
    y_train,
    X_val_trend,
    y_val,
    "P2_E5_Trend_Aware"
)

Epoch 1/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - loss: 5383.5757 - mae: 64.1275 - val_loss: 3249.3442 - val_mae: 50.4494
Epoch 2/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - loss: 2459.3511 - mae: 44.7949 - val_loss: 1921.0908 - val_mae: 40.0278
Epoch 3/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 1797.1567 - mae: 38.5198 - val_loss: 1738.3641 - val_mae: 37.2937
Epoch 4/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 1720.5663 - mae: 36.9440 - val_loss: 1734.0259 - val_mae: 36.7562
Epoch 5/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - loss: 1619.6049 - mae: 35.5699 - val_loss: 902.6616 - val_mae: 26.3046
Epoch 6/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 533.8818 - mae: 19.3389 - val_loss: 452.0300 - val_mae: 17.2687
Epoch 7/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 355.3239 - mae: 14.3207 - val_loss: 441.2208 - val_mae: 16.0658
Epoch 8/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - loss: 302.2217 - mae: 12.5680 - val_loss: 449.7375 - val_mae: 15.7356

In [ ]:
pred_p2e5, global_p2e5, stage_p2e5 = evaluate_model(
    p2e5_model,
    X_val_trend,
    y_val,
    label="P2-E5 Trend Aware"
)

print("\nGLOBAL PERFORMANCE")
display(global_p2e5)

print("\nLIFE-STAGE PERFORMANCE")
display(stage_p2e5)


GLOBAL PERFORMANCE


,Model,N,RMSE,MAE,Bias_pred_minus_true
0,P2-E5 Trend Aware,10332,19.859728,14.414912,-3.506385



LIFE-STAGE PERFORMANCE


,Model,Life_Stage,N,RMSE,MAE,Bias_pred_minus_true
0,P2-E5 Trend Aware,Early,6363,19.613069,14.429774,-9.921436
1,P2-E5 Trend Aware,Mid,1960,26.091819,20.844353,8.990522
2,P2-E5 Trend Aware,Late,2009,12.077118,8.095221,4.619563


In [ ]:
comparison_p2e5 = pd.concat(
    [
        stage_baseline,
        stage_p2e5
    ],
    ignore_index=True
)

pivot_p2e5 = comparison_p2e5.pivot(
    index="Life_Stage",
    columns="Model",
    values="RMSE"
)

display(pivot_p2e5)

Model,P2-E1 Frozen Paper1 Baseline,P2-E5 Trend Aware
Life_Stage,,
Early,16.331137,19.613069
Late,10.839741,12.077118
Mid,27.739052,26.091819


# 7. P2-E6 — Trend-based residual degradation correction

## Objective

Preserve the frozen P2-E1 baseline predictor and train a smaller trend-aware network to predict a correction:

**final RUL = baseline RUL + learned correction**

This tests whether trend information is more useful as a localized correction than as a replacement for the original representation.

### Result

- Global RMSE: **18.332**
- Early RMSE: **17.154**
- Mid RMSE: **26.659**
- Late RMSE: **10.143**

The correction helped Mid and Late slightly, but worsened Early and global performance. More importantly, the learned correction was almost constant across life stages (approximately -2 cycles), suggesting weak residual structure learning.

**Decision: PARTIAL / not promoted.**

A methodological concern was also identified: the first residual model used in-sample baseline residuals on the training set. P2-E6b was therefore added to test out-of-fold residual targets.

In [ ]:
# ============================================================
# P2-E6 — FREEZE BASELINE
# ============================================================

baseline_model.trainable = False

baseline_train_pred = baseline_model.predict(
    X_train,
    verbose=0
).reshape(-1)

baseline_val_pred = baseline_model.predict(
    X_val,
    verbose=0
).reshape(-1)

# Residual that the correction model needs to learn
residual_train = y_train.reshape(-1) - baseline_train_pred
residual_val = y_val.reshape(-1) - baseline_val_pred

print("Baseline train residual mean:", residual_train.mean())
print("Baseline val residual mean:", residual_val.mean())
print("Baseline val residual std:", residual_val.std())

Baseline train residual mean: -1.5316921
Baseline val residual mean: 0.4324066
Baseline val residual std: 18.244978


In [ ]:
# ============================================================
# P2-E6 — TREND-BASED RESIDUAL CORRECTOR
# ============================================================

def build_residual_corrector(input_dim=34):

    inp = Input(
        shape=(WINDOW_SIZE, input_dim),
        name="trend_input"
    )

    x = Dense(
        12,
        use_bias=False,
        name="trend_projection"
    )(inp)

    x = LSTM(
        32,
        name="trend_encoder"
    )(x)

    correction = Dense(
        1,
        name="rul_correction"
    )(x)

    model = Model(
        inp,
        correction,
        name="P2_E6_Residual_Degradation_Corrector"
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(5e-4),
        loss="mse",
        metrics=["mae"]
    )

    return model

In [ ]:
set_all_seeds(SEED)

residual_model = build_residual_corrector(
    input_dim=X_train_trend.shape[-1]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=10,
        restore_best_weights=True,
        verbose=1
    )
]

history_residual = residual_model.fit(
    X_train_trend,
    residual_train,

    validation_data=(
        X_val_trend,
        residual_val
    ),

    epochs=50,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - loss: 205.6113 - mae: 9.8427 - val_loss: 337.8811 - val_mae: 13.1444
Epoch 2/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 204.7709 - mae: 9.9215 - val_loss: 337.8885 - val_mae: 13.1959
Epoch 3/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - loss: 204.2309 - mae: 9.9446 - val_loss: 336.7628 - val_mae: 13.1995
Epoch 4/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 203.8201 - mae: 9.9408 - val_loss: 336.0548 - val_mae: 13.1944
Epoch 5/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 203.2813 - mae: 9.9304 - val_loss: 336.8310 - val_mae: 13.2326
Epoch 6/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 202.3910 - mae: 9.9102 - val_loss: 336.4027 - val_mae: 13.1506
Epoch 7/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 200.9370 - mae: 9.8781 - val_loss: 337.0531 - val_mae: 13.1316
Epoch 8/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 199.1680 - mae: 9.8448 - val_loss: 339.0758 - val_mae: 13.1808
Epoch 9/50
683/6

In [ ]:
# ============================================================
# P2-E6 — FINAL CORRECTED PREDICTIONS
# ============================================================

correction_val = residual_model.predict(
    X_val_trend,
    verbose=0
).reshape(-1)

pred_p2e6 = baseline_val_pred + correction_val

global_p2e6 = pd.DataFrame([{
    "Model": "P2-E6 Residual Trend Correction",
    **regression_metrics(y_val, pred_p2e6)
}])

stage_p2e6 = evaluate_by_life_stage(
    y_val,
    pred_p2e6
)

stage_p2e6.insert(
    0,
    "Model",
    "P2-E6 Residual Trend Correction"
)

print("GLOBAL PERFORMANCE")
display(global_p2e6)

print("\nLIFE-STAGE PERFORMANCE")
display(stage_p2e6)

GLOBAL PERFORMANCE


,Model,N,RMSE,MAE,Bias_pred_minus_true
0,P2-E6 Residual Trend Correction,10332,18.331799,13.194449,-2.529145



LIFE-STAGE PERFORMANCE


,Model,Life_Stage,N,RMSE,MAE,Bias_pred_minus_true
0,P2-E6 Residual Trend Correction,Early,6363,17.154402,12.731514,-8.191943
1,P2-E6 Residual Trend Correction,Mid,1960,26.658905,21.121834,10.814980
2,P2-E6 Residual Trend Correction,Late,2009,10.143080,6.926643,2.387677


In [ ]:
comparison_p2e6 = pd.concat(
    [
        stage_baseline,
        stage_p2e5,
        stage_p2e6
    ],
    ignore_index=True
)

pivot_p2e6 = comparison_p2e6.pivot(
    index="Life_Stage",
    columns="Model",
    values="RMSE"
)

display(pivot_p2e6)

Model,P2-E1 Frozen Paper1 Baseline,P2-E5 Trend Aware,P2-E6 Residual Trend Correction
Life_Stage,,,
Early,16.331137,19.613069,17.154402
Late,10.839741,12.077118,10.143080
Mid,27.739052,26.091819,26.658905


In [ ]:
print("Correction statistics")
print("---------------------")
print("Mean:", np.mean(correction_val))
print("Std :", np.std(correction_val))
print("Min :", np.min(correction_val))
print("Max :", np.max(correction_val))

for stage in ["Early", "Mid", "Late"]:
    mask = assign_life_stage(y_val) == stage

    print(
        stage,
        "mean correction:",
        correction_val[mask].mean()
    )

Correction statistics
---------------------
Mean: -2.0967388
Std : 1.0003248
Min : -3.7755175
Max : 1.6429869
Early mean correction: -2.216201
Mid mean correction: -2.0513356
Late mean correction: -1.7626681


# 8. P2-E6b — Out-of-fold residual correction

## Objective

Remove the in-sample stacking mismatch from P2-E6 by generating out-of-fold baseline predictions for the training set, then fitting the same residual corrector to those OOF residuals.

### Result

- Global RMSE: **18.229**
- Early RMSE: **16.238**
- Mid RMSE: **27.726**
- Late RMSE: **11.126**
- Correlation between true validation residual and learned correction: **0.044**

The tiny global improvement is not accompanied by meaningful repair of the mid-life failure. The correction has almost no predictive relationship with the true residual.

**Decision: FAIL / residual-correction mechanism not promoted.**

### Important methodological note

The OOF folds used here were random K-fold splits of windows. If multiple windows from the same engine can occur in different folds, the OOF RMSE values around 9–10 cycles are **not directly comparable** with the held-out validation RMSE and should not be reported as a generalization claim. Any future OOF analysis should use engine-grouped folds when engine IDs are available.

In [ ]:
from sklearn.model_selection import KFold

N_FOLDS = 5

kf = KFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED
)

oof_baseline_pred = np.zeros(
    len(y_train),
    dtype=np.float32
)

for fold, (fit_idx, hold_idx) in enumerate(
    kf.split(X_train),
    start=1
):
    print(f"\n{'='*60}")
    print(f"OOF BASELINE — FOLD {fold}/{N_FOLDS}")
    print(f"{'='*60}")

    tf.keras.backend.clear_session()
    set_all_seeds(SEED + fold)

    fold_model = build_frozen_point_baseline()

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            mode="min",
            patience=10,
            restore_best_weights=True,
            verbose=0
        )
    ]

    fold_model.fit(
        X_train[fit_idx],
        y_train[fit_idx],

        validation_data=(
            X_train[hold_idx],
            y_train[hold_idx]
        ),

        epochs=50,
        batch_size=64,
        callbacks=callbacks,
        verbose=0
    )

    oof_baseline_pred[hold_idx] = (
        fold_model.predict(
            X_train[hold_idx],
            verbose=0
        ).reshape(-1)
    )

    fold_rmse = np.sqrt(
        mean_squared_error(
            y_train[hold_idx],
            oof_baseline_pred[hold_idx]
        )
    )

    print(f"Fold {fold} RMSE: {fold_rmse:.4f}")


OOF BASELINE — FOLD 1/5
Fold 1 RMSE: 9.4450

OOF BASELINE — FOLD 2/5
Fold 2 RMSE: 9.4373

OOF BASELINE — FOLD 3/5
Fold 3 RMSE: 10.3775

OOF BASELINE — FOLD 4/5
Fold 4 RMSE: 10.6336

OOF BASELINE — FOLD 5/5
Fold 5 RMSE: 10.1382


In [ ]:
oof_residual_train = (
    y_train.reshape(-1)
    - oof_baseline_pred
)

print("\nOOF residual statistics")
print("-----------------------")
print("Mean:", oof_residual_train.mean())
print("Std :", oof_residual_train.std())

oof_rmse = np.sqrt(
    mean_squared_error(
        y_train,
        oof_baseline_pred
    )
)

print("OOF baseline RMSE:", oof_rmse)


OOF residual statistics
-----------------------
Mean: 0.013049131
Std : 10.018178
OOF baseline RMSE: 10.018186435091044


In [ ]:
tf.keras.backend.clear_session()
set_all_seeds(SEED)

residual_model_oof = build_residual_corrector(
    input_dim=X_train_trend.shape[-1]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=10,
        restore_best_weights=True,
        verbose=1
    )
]

history_residual_oof = residual_model_oof.fit(
    X_train_trend,
    oof_residual_train,

    validation_data=(
        X_val_trend,
        residual_val
    ),

    epochs=50,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - loss: 100.2190 - mae: 6.5860 - val_loss: 334.0237 - val_mae: 12.6928
Epoch 2/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 100.1109 - mae: 6.5942 - val_loss: 334.1423 - val_mae: 12.7140
Epoch 3/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - loss: 100.0051 - mae: 6.5952 - val_loss: 333.6362 - val_mae: 12.7214
Epoch 4/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 99.8634 - mae: 6.5967 - val_loss: 333.3853 - val_mae: 12.7278
Epoch 5/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 99.7361 - mae: 6.5998 - val_loss: 333.3260 - val_mae: 12.7375
Epoch 6/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - loss: 99.5912 - mae: 6.6022 - val_loss: 333.3587 - val_mae: 12.7457
Epoch 7/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 99.4209 - mae: 6.6037 - val_loss: 333.3906 - val_mae: 12.7536
Epoch 8/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 99.2618 - mae: 6.6055 - val_loss: 333.0565 - val_mae: 12.7484
Epoch 9/50
683/683 ━━

In [ ]:
correction_val_oof = (
    residual_model_oof.predict(
        X_val_trend,
        verbose=0
    ).reshape(-1)
)

pred_p2e6b = (
    baseline_val_pred
    + correction_val_oof
)

global_p2e6b = pd.DataFrame([{
    "Model": "P2-E6b OOF Residual Correction",
    **regression_metrics(
        y_val,
        pred_p2e6b
    )
}])

stage_p2e6b = evaluate_by_life_stage(
    y_val,
    pred_p2e6b
)

stage_p2e6b.insert(
    0,
    "Model",
    "P2-E6b OOF Residual Correction"
)

print("GLOBAL PERFORMANCE")
display(global_p2e6b)

print("\nLIFE-STAGE PERFORMANCE")
display(stage_p2e6b)

GLOBAL PERFORMANCE


,Model,N,RMSE,MAE,Bias_pred_minus_true
0,P2-E6b OOF Residual Correction,10332,18.228699,12.695736,-0.101415



LIFE-STAGE PERFORMANCE


,Model,Life_Stage,N,RMSE,MAE,Bias_pred_minus_true
0,P2-E6b OOF Residual Correction,Early,6363,16.238085,11.385408,-5.730718
1,P2-E6b OOF Residual Correction,Mid,1960,27.726176,22.004379,13.260982
2,P2-E6b OOF Residual Correction,Late,2009,11.125660,7.764269,4.691496


In [ ]:
print("Correction statistics")
print("---------------------")

print("Mean:", correction_val_oof.mean())
print("Std :", correction_val_oof.std())
print("Min :", correction_val_oof.min())
print("Max :", correction_val_oof.max())

stages = assign_life_stage(y_val)

for stage in ["Early", "Mid", "Late"]:
    mask = stages == stage

    print(
        stage,
        "mean correction:",
        correction_val_oof[mask].mean(),
        "std:",
        correction_val_oof[mask].std()
    )

Correction statistics
---------------------
Mean: 0.33099163
Std : 1.0232818
Min : -4.643677
Max : 3.449848
Early mean correction: 0.24502397 std: 1.041158
Mid mean correction: 0.39466602 std: 0.98201305
Late mean correction: 0.5411511 std: 0.96924156


In [ ]:
true_val_residual = (
    y_val.reshape(-1)
    - baseline_val_pred
)

corr = np.corrcoef(
    true_val_residual,
    correction_val_oof
)[0, 1]

print(
    "Correlation between true residual "
    "and learned correction:",
    corr
)

Correlation between true residual and learned correction: 0.044219834581244705


P2-E6b — FAIL / RESIDUAL CORRECTION NOT PROMOTED

Global RMSE:
P2-E1 = 18.2501
P2-E6b = 18.2287

Mid-life RMSE:
P2-E1 = 27.7391
P2-E6b = 27.7262

Residual-correction correlation:
r = 0.0442

Finding:
Out-of-fold residual training removes the previous in-sample
stacking issue, but the learned correction has almost no predictive
relationship with the true validation residual.

Decision:
Reject trend-based residual correction as the main Paper 2 mechanism.

# 9. P2-E7 — Cycle-to-cycle first-difference augmentation

## Objective

Test whether preserving the full within-window temporal change pattern is more useful than summarizing each sensor with one slope.

Input representation:

- 14 sensor levels
- 14 first-difference channels, Δx(t) = x(t) - x(t-1)
- 6 regime one-hot channels

Total: **34 channels**.

This experiment directly tests whether short-horizon local temporal derivatives reveal the missing mid-life transition dynamics.

In [ ]:
def make_delta_aware_inputs(X_raw):
    sensors = X_raw[:, :, :N_SENSORS].astype(np.float32)
    regime = X_raw[:, :, N_SENSORS].astype(int)

    regime_oh = tf.keras.utils.to_categorical(
        regime,
        num_classes=N_REGIMES
    ).astype(np.float32)

    # First temporal difference
    delta = np.zeros_like(sensors)
    delta[:, 1:, :] = sensors[:, 1:, :] - sensors[:, :-1, :]

    X_delta = np.concatenate(
        [sensors, delta, regime_oh],
        axis=-1
    )

    return X_delta.astype(np.float32)

In [ ]:
X_train_delta = make_delta_aware_inputs(X_train_raw)
X_val_delta = make_delta_aware_inputs(X_val_raw)

print(X_train_delta.shape)
# expected: (43696, 30, 34)

(43696, 30, 34)


In [ ]:
def build_delta_aware_model(input_dim=34):
    inp = Input(
        shape=(WINDOW_SIZE, input_dim),
        name="delta_aware_input"
    )

    x = Dense(
        PROJECTION_DIM,
        use_bias=False,
        name="projection"
    )(inp)

    x = LSTM(
        64,
        name="lstm_core"
    )(x)

    out = Dense(1, name="rul")(x)

    model = Model(
        inp,
        out,
        name="P2_E7_Delta_Aware_LSTM"
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="mse",
        metrics=["mae"]
    )

    return model

In [ ]:
# ============================================================
# P2-E7 — BUILD DELTA-AWARE INPUTS
# ============================================================

X_train_delta = make_delta_aware_inputs(X_train_raw)
X_val_delta = make_delta_aware_inputs(X_val_raw)

print("Train delta shape:", X_train_delta.shape)
print("Val delta shape  :", X_val_delta.shape)

# Expected:
# Train: (43696, 30, 34)
# Val  : (10332, 30, 34)

Train delta shape: (43696, 30, 34)
Val delta shape  : (10332, 30, 34)


In [ ]:
# ============================================================
# P2-E7 — TRAIN DELTA-AWARE MODEL
# ============================================================

set_all_seeds(SEED)

p2e7_model = build_delta_aware_model(
    input_dim=X_train_delta.shape[-1]
)

p2e7_model.summary()

history_p2e7, p2e7_ckpt = fit_model(
    p2e7_model,
    X_train_delta,
    y_train,
    X_val_delta,
    y_val,
    "P2_E7_Delta_Aware"
)

print(
    "Training stopped after:",
    len(history_p2e7.history["loss"]),
    "epochs"
)

Model: "P2_E7_Delta_Aware_LSTM"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ delta_aware_input (InputLayer)  │ (None, 30, 34)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ projection (Dense)              │ (None, 30, 18)         │           612 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_core (LSTM)                │ (None, 64)             │        21,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rul (Dense)                     │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,925 (85.64 KB)

 Trainable params: 21,925 (85.64 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - loss: 5323.8457 - mae: 63.7548 - val_loss: 3192.4631 - val_mae: 50.0595
Epoch 2/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - loss: 2441.6748 - mae: 44.6589 - val_loss: 1917.5638 - val_mae: 39.9905
Epoch 3/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 1795.7974 - mae: 38.5007 - val_loss: 1738.2214 - val_mae: 37.2878
Epoch 4/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 1720.5074 - mae: 36.9411 - val_loss: 1734.0370 - val_mae: 36.7554
Epoch 5/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - loss: 1560.3805 - mae: 34.8729 - val_loss: 850.2777 - val_mae: 25.8725
Epoch 6/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 551.1706 - mae: 19.7276 - val_loss: 467.3196 - val_mae: 17.3303
Epoch 7/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - loss: 373.0947 - mae: 14.6900 - val_loss: 413.2076 - val_mae: 15.4630
Epoch 8/50
683/683 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 320.9187 - mae: 12.9376 - val_loss: 445.5994 - val_mae: 15.4917

In [ ]:
# ============================================================
# P2-E7 — EVALUATION
# ============================================================

pred_p2e7, global_p2e7, stage_p2e7 = evaluate_model(
    p2e7_model,
    X_val_delta,
    y_val,
    label="P2-E7 Delta Aware"
)

print("\n" + "="*60)
print("P2-E7 GLOBAL VALIDATION PERFORMANCE")
print("="*60)
display(global_p2e7)

print("\n" + "="*60)
print("P2-E7 PERFORMANCE BY LIFE STAGE")
print("="*60)
display(stage_p2e7)


P2-E7 GLOBAL VALIDATION PERFORMANCE


,Model,N,RMSE,MAE,Bias_pred_minus_true
0,P2-E7 Delta Aware,10332,19.876207,13.56156,0.741187



P2-E7 PERFORMANCE BY LIFE STAGE


,Model,Life_Stage,N,RMSE,MAE,Bias_pred_minus_true
0,P2-E7 Delta Aware,Early,6363,16.882244,11.419859,-4.815502
1,P2-E7 Delta Aware,Mid,1960,31.117765,25.360327,14.121833
2,P2-E7 Delta Aware,Late,2009,13.577922,8.833858,5.286305


In [ ]:
# ============================================================
# P2-E1 vs P2-E5 vs P2-E7
# ============================================================

comparison_p2e7 = pd.concat(
    [
        stage_baseline,
        stage_p2e5,
        stage_p2e7
    ],
    ignore_index=True
)

pivot_p2e7 = comparison_p2e7.pivot(
    index="Life_Stage",
    columns="Model",
    values="RMSE"
)

display(pivot_p2e7)

Model,P2-E1 Frozen Paper1 Baseline,P2-E5 Trend Aware,P2-E7 Delta Aware
Life_Stage,,,
Early,16.331137,19.613069,16.882244
Late,10.839741,12.077118,13.577922
Mid,27.739052,26.091819,31.117765


In [ ]:
baseline_rmse = stage_baseline.set_index("Life_Stage")["RMSE"]
delta_rmse = stage_p2e7.set_index("Life_Stage")["RMSE"]

p2e7_improvement = pd.DataFrame({
    "P2-E1 Baseline RMSE": baseline_rmse,
    "P2-E7 Delta RMSE": delta_rmse
})

p2e7_improvement["RMSE Change"] = (
    p2e7_improvement["P2-E7 Delta RMSE"]
    - p2e7_improvement["P2-E1 Baseline RMSE"]
)

p2e7_improvement["Improvement %"] = (
    (
        p2e7_improvement["P2-E1 Baseline RMSE"]
        - p2e7_improvement["P2-E7 Delta RMSE"]
    )
    / p2e7_improvement["P2-E1 Baseline RMSE"]
    * 100
)

display(p2e7_improvement)

,P2-E1 Baseline RMSE,P2-E7 Delta RMSE,RMSE Change,Improvement %
Life_Stage,,,,
Early,16.331137,16.882244,0.551107,-3.374580
Mid,27.739052,31.117765,3.378713,-12.180347
Late,10.839741,13.577922,2.738180,-25.260568


P2-E7 — FAIL / DO NOT PROMOTE

Global RMSE:
18.2501 -> 19.8762

Mid-life RMSE:
27.7391 -> 31.1178

Finding:
Explicit cycle-to-cycle sensor differences degrade performance,
including the target mid-life region.

Interpretation:
Local first differences are likely too noisy and/or too short-horizon
to represent the degradation transition responsible for the Paper 1
mid-life failure.

Decision:
Reject raw first-difference augmentation.

In [ ]:
# Search your Drive for likely baseline-generation files
import os

root = "/content/drive/MyDrive"

for dirpath, dirnames, filenames in os.walk(root):
    for f in filenames:
        name = f.lower()

        if (
            "fd004" in name
            and (
                "baseline" in name
                or "tensor" in name
                or "preprocess" in name
                or "split" in name
            )
        ):
            print(os.path.join(dirpath, f))

/content/drive/MyDrive/Result Table 01_FD004_Research_Baseline.gsheet
/content/drive/MyDrive/01_FD004_Research_Baseline.gdoc
/content/drive/MyDrive/FD004_Research/Notebooks/01_FD004_Research_Baseline.ipynb
/content/drive/MyDrive/FD004_Research/Notebooks/02_FD004_Research_Baseline.ipynb
/content/drive/MyDrive/FD004_Research/artifacts/FD004_baseline_tensors.npz
/content/drive/MyDrive/FD004_Research/Output/01_FD004_Research_Baseline.pdf


In [ ]:
import os
import json

ROOT = "/content/drive/MyDrive/FD004_Research"

targets = [
    "FD004_baseline_tensors.npz",
    "np.savez",
    "np.savez_compressed"
]

for dirpath, _, filenames in os.walk(ROOT):
    for filename in filenames:
        if not filename.endswith(".ipynb"):
            continue

        path = os.path.join(dirpath, filename)

        try:
            with open(path, "r", encoding="utf-8") as f:
                nb = json.load(f)

            text = "\n".join(
                "".join(cell.get("source", []))
                for cell in nb.get("cells", [])
            )

            matches = [t for t in targets if t in text]

            if matches:
                print("\nFOUND:", path)
                print("Matches:", matches)

        except Exception as e:
            pass


FOUND: /content/drive/MyDrive/FD004_Research/Notebooks/01_FD004_Research_Baseline.ipynb
Matches: ['FD004_baseline_tensors.npz', 'np.savez', 'np.savez_compressed']

FOUND: /content/drive/MyDrive/FD004_Research/Notebooks/06_FD004_Paper2_Degradation_Aware_Research_Starter.ipynb
Matches: ['FD004_baseline_tensors.npz']


In [ ]:
import json

NB_PATH = "/content/drive/MyDrive/FD004_Research/Notebooks/01_FD004_Research_Baseline.ipynb"

with open(NB_PATH, "r", encoding="utf-8") as f:
    nb = json.load(f)

targets = [
    "FD004_baseline_tensors.npz",
    "np.savez",
    "np.savez_compressed",
    "train_test_split",
    "StandardScaler",
    "train_FD004.txt",
    "WINDOW_SIZE"
]

for i, cell in enumerate(nb["cells"]):
    src = "".join(cell.get("source", []))

    if any(t in src for t in targets):
        print("\n" + "=" * 80)
        print("CELL", i, "|", cell.get("cell_type"))
        print("=" * 80)
        print(src)


CELL 3 | code
import random
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# --- Seed & Determinism ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# --- Pipeline Hyperparameters ---
DATA_DIR = '/content/drive/MyDrive/CMAPSSData'
RUL_CLIP = 125
WINDOW_SIZE = 30
N_REGIMES = 6
VAL_SPLIT_RATIO = 0.20

# 14 standard informative sensors for C-MAPSS FD002/FD004 (dropping constant/low-variance sensors)
OPERATING_SETTINGS = ["setting_1", "setting_2", "setting_3"]
SELECTED_SENSORS = [
    "s_2", "s_3", "s_4", "s_7", "s_8", "s_9",
    "s_11", "s_12", "s_13", "s_14", "s_15", "s_17", "s_20", "s_21"
]
ALL_RAW_COLS = (
    ["unit_id", "time_cycles"]
    + OPERATING_SETTINGS
    + [f"s_{i}" for i in range(1, 22)]
)

CELL 5 | code
# Point directly to your mounted Google Drive directory
DATA_DIR = '/content/drive/MyDrive/CMAPSSData'

train_df_raw = pd.read_csv(
    os.path.join(D

In [ ]:
print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)

NameError: name 'X_train' is not defined

### Review correction applied to P2-E8 execution path

This reviewed copy closes three reproducibility risks in the regenerated notebook:

- the preflight-resolved raw-data directory is preserved instead of being overwritten by a hard-coded Colab path;
- the mandatory 30-cycle gate checks numerical equality with the frozen Paper 1 tensor artifact, not shapes alone;
- the matched-endpoint comparison explicitly uses the trained P2-E1 best checkpoint (or a verified trained in-memory model), preventing accidental comparison against an untrained baseline.

No experimental hyperparameter or model architecture is changed by these corrections.


# 10. P2-E8 — Controlled 60-cycle longer-context experiment

## Why this experiment is next

P2-E2 showed that degradation stage is highly informative when known perfectly, but P2-E3 through P2-E7 did not recover that benefit reliably from engineered features inside the frozen 30-cycle representation.

P2-E8 tests a narrower hypothesis:

> **The 30-cycle observation window may be too short to infer degradation state robustly. A longer temporal context may improve the mid-life failure region without handcrafted stage or derivative features.**

### Controlled change

Only temporal context changes:

- P2-E1: **30 cycles**
- P2-E8: **60 cycles**

Everything else remains frozen: seed 42, 80/20 engine split, RUL clip 125, 14 selected sensors, six operating regimes, train-only preprocessing, one-hot regime representation, 20→18 projection, LSTM(64), Adam 1e-3, and MSE.

### Mandatory integrity gate

Before building 60-cycle windows, reconstruct the Paper 1 preprocessing from raw FD004 and regenerate the 30-cycle tensors. The dimensions must match the frozen artifact exactly:

- `X_train = (43696, 30, 15)`
- `X_val   = (10332, 30, 15)`
- `X_test  = (248, 30, 15)`

If any assertion fails, stop. Do not train P2-E8 until the reconstruction is verified.

### Fair-comparison note

The 60-cycle validation set contains fewer endpoints than the 30-cycle validation set. A final P2-E1 vs P2-E8 claim therefore requires a matched-endpoint comparison after training.


## P2-E8 execution checkpoint

This working copy begins the next project step. Before reconstruction/training, **both** the frozen Paper 1 raw FD004 files and the frozen `FD004_baseline_tensors.npz` artifact must be available. The preflight below does not alter any experimental choice; it only resolves either the original Colab paths or explicitly uploaded local copies.

The 30-cycle integrity gate is deliberately strict: it checks the expected shapes **and** verifies reconstructed `X/y` tensors numerically against the frozen artifact. P2-E8 must stop if that fidelity check fails.


In [ ]:
# ============================================================
# P2-E8.0 — EXECUTION PREFLIGHT (NO MODEL CHANGES)
# ============================================================
from pathlib import Path

required_raw = ["train_FD004.txt", "test_FD004.txt", "RUL_FD004.txt"]
candidate_data_dirs = [
    Path("/content/drive/MyDrive/CMAPSSData"),
    Path("/mnt/data"),
]

resolved_data_dir = None
for d in candidate_data_dirs:
    if all((d / name).exists() for name in required_raw):
        resolved_data_dir = d
        break

baseline_candidates = [
    Path(DATA_PATH),
    Path("/mnt/data/FD004_baseline_tensors.npz"),
]
resolved_baseline = next((p for p in baseline_candidates if p.exists()), None)

print("Raw FD004 directory:", resolved_data_dir)
print("Frozen baseline tensor artifact:", resolved_baseline)

missing = []
if resolved_data_dir is None:
    missing.append(
        "train_FD004.txt, test_FD004.txt and RUL_FD004.txt "
        "(kept together in /content/drive/MyDrive/CMAPSSData or /mnt/data)"
    )
if resolved_baseline is None:
    missing.append(
        "FD004_baseline_tensors.npz "
        "(/content/drive/MyDrive/FD004_Research/artifacts or /mnt/data)"
    )

if missing:
    raise FileNotFoundError(
        "P2-E8 preflight failed. Missing required frozen artifact(s):\n- "
        + "\n- ".join(missing)
    )

P2E8_DATA_DIR = str(resolved_data_dir)
DATA_PATH = Path(resolved_baseline)

print("PASS: P2-E8 data preflight complete.")
print("Resolved raw directory:", P2E8_DATA_DIR)
print("Resolved frozen tensor artifact:", DATA_PATH)


Raw FD004 directory: /content/drive/MyDrive/CMAPSSData
Frozen baseline tensor artifact: /content/drive/MyDrive/FD004_Research/artifacts/FD004_baseline_tensors.npz
PASS: P2-E8 data preflight complete.
Resolved raw directory: /content/drive/MyDrive/CMAPSSData
Resolved frozen tensor artifact: /content/drive/MyDrive/FD004_Research/artifacts/FD004_baseline_tensors.npz


In [ ]:
# ============================================================
# P2-E8.1 — RAW FD004 RECONSTRUCTION
# ============================================================

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# P2E8_DATA_DIR is resolved by the preflight cell above.
if "P2E8_DATA_DIR" not in globals():
    raise RuntimeError("Run P2-E8.0 preflight before reconstruction.")
P2E8_RUL_CLIP = 125
P2E8_N_REGIMES = 6
P2E8_VAL_SPLIT_RATIO = 0.20

P2E8_OPERATING_SETTINGS = ["setting_1", "setting_2", "setting_3"]

P2E8_SELECTED_SENSORS = [
    "s_2", "s_3", "s_4", "s_7", "s_8", "s_9",
    "s_11", "s_12", "s_13", "s_14",
    "s_15", "s_17", "s_20", "s_21"
]

P2E8_ALL_RAW_COLS = (
    ["unit_id", "time_cycles"]
    + P2E8_OPERATING_SETTINGS
    + [f"s_{i}" for i in range(1, 22)]
)

p2e8_train_raw = pd.read_csv(
    os.path.join(P2E8_DATA_DIR, "train_FD004.txt"),
    sep=r"\s+",
    header=None,
    names=P2E8_ALL_RAW_COLS
)

p2e8_test_raw = pd.read_csv(
    os.path.join(P2E8_DATA_DIR, "test_FD004.txt"),
    sep=r"\s+",
    header=None,
    names=P2E8_ALL_RAW_COLS
)

p2e8_rul_test_raw = pd.read_csv(
    os.path.join(P2E8_DATA_DIR, "RUL_FD004.txt"),
    sep=r"\s+",
    header=None,
    names=["true_rul"]
)

print("Train raw:", p2e8_train_raw.shape,
      "| engines:", p2e8_train_raw["unit_id"].nunique())
print("Test raw :", p2e8_test_raw.shape,
      "| engines:", p2e8_test_raw["unit_id"].nunique())
print("RUL file :", p2e8_rul_test_raw.shape)

assert not p2e8_train_raw.isnull().values.any()
assert not p2e8_test_raw.isnull().values.any()
assert len(p2e8_rul_test_raw) == p2e8_test_raw["unit_id"].nunique()


Train raw: (61249, 26) | engines: 249
Test raw : (41214, 26) | engines: 248
RUL file : (248, 1)


In [ ]:
# ============================================================
# P2-E8.2 — RUL TARGETS
# ============================================================

p2e8_train_processed = p2e8_train_raw.copy()

p2e8_max_cycles = (
    p2e8_train_processed
    .groupby("unit_id")["time_cycles"]
    .max()
    .rename("max_cycle")
)

p2e8_train_processed = p2e8_train_processed.merge(
    p2e8_max_cycles,
    on="unit_id",
    how="left"
)

p2e8_train_processed["RUL"] = (
    p2e8_train_processed["max_cycle"]
    - p2e8_train_processed["time_cycles"]
).clip(upper=P2E8_RUL_CLIP).astype(np.float32)

# RUL_FD004.txt contains one terminal target per test engine in unit order.
p2e8_test_targets = p2e8_rul_test_raw.copy()
p2e8_test_targets.insert(
    0,
    "unit_id",
    np.arange(1, len(p2e8_test_targets) + 1, dtype=int)
)
p2e8_test_targets["true_rul"] = (
    p2e8_test_targets["true_rul"]
    .clip(upper=P2E8_RUL_CLIP)
    .astype(np.float32)
)

print("Train RUL range:",
      p2e8_train_processed["RUL"].min(),
      "to",
      p2e8_train_processed["RUL"].max())


Train RUL range: 0.0 to 125.0


In [ ]:
# ============================================================
# P2-E8.3 — EXACT ENGINE-LEVEL 80/20 SPLIT
# ============================================================

p2e8_unique_engines = p2e8_train_processed["unit_id"].unique()

p2e8_rng = np.random.RandomState(SEED)
p2e8_shuffled_engines = p2e8_rng.permutation(p2e8_unique_engines)

p2e8_n_val = int(
    len(p2e8_shuffled_engines) * P2E8_VAL_SPLIT_RATIO
)

p2e8_val_engine_ids = p2e8_shuffled_engines[:p2e8_n_val]
p2e8_train_engine_ids = p2e8_shuffled_engines[p2e8_n_val:]

assert set(p2e8_train_engine_ids).isdisjoint(
    set(p2e8_val_engine_ids)
)

p2e8_train_split_df = p2e8_train_processed[
    p2e8_train_processed["unit_id"].isin(p2e8_train_engine_ids)
].copy()

p2e8_val_split_df = p2e8_train_processed[
    p2e8_train_processed["unit_id"].isin(p2e8_val_engine_ids)
].copy()

p2e8_test_split_df = p2e8_test_raw.copy()

print(
    "Engines -> Train:", len(p2e8_train_engine_ids),
    "| Val:", len(p2e8_val_engine_ids),
    "| Test:", p2e8_test_split_df["unit_id"].nunique()
)


Engines -> Train: 200 | Val: 49 | Test: 248


In [ ]:
# ============================================================
# P2-E8.4 — TRAIN-ONLY PREPROCESSING
# ============================================================

p2e8_setting_scaler = StandardScaler()

p2e8_train_settings_scaled = p2e8_setting_scaler.fit_transform(
    p2e8_train_split_df[P2E8_OPERATING_SETTINGS]
)
p2e8_val_settings_scaled = p2e8_setting_scaler.transform(
    p2e8_val_split_df[P2E8_OPERATING_SETTINGS]
)
p2e8_test_settings_scaled = p2e8_setting_scaler.transform(
    p2e8_test_split_df[P2E8_OPERATING_SETTINGS]
)

p2e8_kmeans = KMeans(
    n_clusters=P2E8_N_REGIMES,
    random_state=SEED,
    n_init=10
)

p2e8_train_split_df["regime"] = p2e8_kmeans.fit_predict(
    p2e8_train_settings_scaled
)

p2e8_val_split_df["regime"] = (
    p2e8_kmeans
    .transform(p2e8_val_settings_scaled)
    .argmin(axis=1)
)

p2e8_test_split_df["regime"] = (
    p2e8_kmeans
    .transform(p2e8_test_settings_scaled)
    .argmin(axis=1)
)

p2e8_sensor_scaler = StandardScaler()

p2e8_train_split_df[P2E8_SELECTED_SENSORS] = (
    p2e8_sensor_scaler.fit_transform(
        p2e8_train_split_df[P2E8_SELECTED_SENSORS]
    )
)

p2e8_val_split_df[P2E8_SELECTED_SENSORS] = (
    p2e8_sensor_scaler.transform(
        p2e8_val_split_df[P2E8_SELECTED_SENSORS]
    )
)

p2e8_test_split_df[P2E8_SELECTED_SENSORS] = (
    p2e8_sensor_scaler.transform(
        p2e8_test_split_df[P2E8_SELECTED_SENSORS]
    )
)

P2E8_FEATURE_COLS = P2E8_SELECTED_SENSORS + ["regime"]

assert p2e8_sensor_scaler.n_samples_seen_ == len(p2e8_train_split_df)
assert p2e8_setting_scaler.n_samples_seen_ == len(p2e8_train_split_df)
assert p2e8_kmeans.cluster_centers_.shape == (
    P2E8_N_REGIMES,
    len(P2E8_OPERATING_SETTINGS)
)

print("Preprocessing isolation checks passed.")


Preprocessing isolation checks passed.


In [ ]:
# ============================================================
# P2-E8.5 — WINDOW GENERATORS + ENDPOINT METADATA
# ============================================================

def p2e8_generate_windows_train_val(
    df,
    window_size,
    feature_cols,
    return_metadata=False
):
    X_list, y_list = [], []
    unit_list, cycle_list = [], []

    for engine_id, group in df.groupby("unit_id"):
        group = group.sort_values("time_cycles")

        features = group[feature_cols].to_numpy()
        ruls = group["RUL"].to_numpy()
        cycles = group["time_cycles"].to_numpy()

        if len(group) < window_size:
            continue

        for t in range(window_size, len(group) + 1):
            X_list.append(features[t-window_size:t])
            y_list.append(ruls[t-1])

            if return_metadata:
                unit_list.append(engine_id)
                cycle_list.append(cycles[t-1])

    X = np.asarray(X_list, dtype=np.float32)
    y = np.asarray(y_list, dtype=np.float32)

    if return_metadata:
        return (
            X, y,
            np.asarray(unit_list),
            np.asarray(cycle_list)
        )

    return X, y


def p2e8_generate_windows_test(
    df,
    rul_df,
    window_size,
    feature_cols,
    clip_val
):
    X_list, y_list = [], []

    for _, row in rul_df.iterrows():
        engine_id = int(row["unit_id"])
        true_rul = min(float(row["true_rul"]), clip_val)

        group = (
            df[df["unit_id"] == engine_id]
            .sort_values("time_cycles")
        )

        features = group[feature_cols].to_numpy()

        if len(features) >= window_size:
            window = features[-window_size:]
        else:
            pad_len = window_size - len(features)
            pad = np.repeat(features[0:1], pad_len, axis=0)
            window = np.vstack([pad, features])

        X_list.append(window)
        y_list.append(true_rul)

    return (
        np.asarray(X_list, dtype=np.float32),
        np.asarray(y_list, dtype=np.float32)
    )


In [ ]:
# ============================================================
# P2-E8.6 — MANDATORY 30-CYCLE INTEGRITY + FIDELITY CHECK
# ============================================================

p2e8_X_train_30_raw, p2e8_y_train_30 = (
    p2e8_generate_windows_train_val(
        p2e8_train_split_df,
        30,
        P2E8_FEATURE_COLS
    )
)

p2e8_X_val_30_raw, p2e8_y_val_30 = (
    p2e8_generate_windows_train_val(
        p2e8_val_split_df,
        30,
        P2E8_FEATURE_COLS
    )
)

p2e8_X_test_30_raw, p2e8_y_test_30 = (
    p2e8_generate_windows_test(
        p2e8_test_split_df,
        p2e8_test_targets,
        30,
        P2E8_FEATURE_COLS,
        P2E8_RUL_CLIP
    )
)

print("Reconstructed frozen 30-cycle tensors")
print("--------------------------------------")
print("X_train:", p2e8_X_train_30_raw.shape)
print("X_val  :", p2e8_X_val_30_raw.shape)
print("X_test :", p2e8_X_test_30_raw.shape)

EXPECTED_TRAIN_30 = (43696, 30, 15)
EXPECTED_VAL_30   = (10332, 30, 15)
EXPECTED_TEST_30  = (248, 30, 15)

assert p2e8_X_train_30_raw.shape == EXPECTED_TRAIN_30, (
    f"STOP: expected {EXPECTED_TRAIN_30}, got {p2e8_X_train_30_raw.shape}"
)
assert p2e8_X_val_30_raw.shape == EXPECTED_VAL_30, (
    f"STOP: expected {EXPECTED_VAL_30}, got {p2e8_X_val_30_raw.shape}"
)
assert p2e8_X_test_30_raw.shape == EXPECTED_TEST_30, (
    f"STOP: expected {EXPECTED_TEST_30}, got {p2e8_X_test_30_raw.shape}"
)

# Shape equality alone is insufficient: compare against the actual frozen artifact.
with np.load(DATA_PATH) as frozen:
    required_keys = ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test"]
    missing_keys = [k for k in required_keys if k not in frozen.files]
    assert not missing_keys, f"STOP: frozen tensor artifact missing keys: {missing_keys}"

    frozen_X_train = frozen["X_train"]
    frozen_y_train = frozen["y_train"]
    frozen_X_val = frozen["X_val"]
    frozen_y_val = frozen["y_val"]
    frozen_X_test = frozen["X_test"]
    frozen_y_test = frozen["y_test"]

# Compare values in the same row/order. Tolerance only accommodates float serialization noise.
np.testing.assert_allclose(p2e8_X_train_30_raw, frozen_X_train, rtol=0, atol=1e-6)
np.testing.assert_allclose(p2e8_y_train_30, frozen_y_train, rtol=0, atol=1e-6)
np.testing.assert_allclose(p2e8_X_val_30_raw, frozen_X_val, rtol=0, atol=1e-6)
np.testing.assert_allclose(p2e8_y_val_30, frozen_y_val, rtol=0, atol=1e-6)
np.testing.assert_allclose(p2e8_X_test_30_raw, frozen_X_test, rtol=0, atol=1e-6)
np.testing.assert_allclose(p2e8_y_test_30, frozen_y_test, rtol=0, atol=1e-6)

print("\nPASS: exact 30-cycle dimensions reproduced.")
print("PASS: reconstructed X/y values match the frozen Paper 1 tensor artifact.")
print("Proceed to P2-E8 60-cycle generation.")


Reconstructed frozen 30-cycle tensors
--------------------------------------
X_train: (43696, 30, 15)
X_val  : (10332, 30, 15)
X_test : (248, 30, 15)

PASS: exact 30-cycle dimensions reproduced.
PASS: reconstructed X/y values match the frozen Paper 1 tensor artifact.
Proceed to P2-E8 60-cycle generation.


In [ ]:
# ============================================================
# P2-E8.7 — TRUE 60-CYCLE TENSORS
# ============================================================

P2E8_WINDOW_SIZE = 60

(
    X_train_60_raw,
    y_train_60,
    train_units_60,
    train_cycles_60
) = p2e8_generate_windows_train_val(
    p2e8_train_split_df,
    P2E8_WINDOW_SIZE,
    P2E8_FEATURE_COLS,
    return_metadata=True
)

(
    X_val_60_raw,
    y_val_60,
    val_units_60,
    val_cycles_60
) = p2e8_generate_windows_train_val(
    p2e8_val_split_df,
    P2E8_WINDOW_SIZE,
    P2E8_FEATURE_COLS,
    return_metadata=True
)

X_test_60_raw, y_test_60 = p2e8_generate_windows_test(
    p2e8_test_split_df,
    p2e8_test_targets,
    P2E8_WINDOW_SIZE,
    P2E8_FEATURE_COLS,
    P2E8_RUL_CLIP
)

print("60-cycle tensors")
print("----------------")
print("Train:", X_train_60_raw.shape, y_train_60.shape)
print("Val  :", X_val_60_raw.shape, y_val_60.shape)
print("Test :", X_test_60_raw.shape, y_test_60.shape)

assert X_train_60_raw.shape[1:] == (60, 15)
assert X_val_60_raw.shape[1:] == (60, 15)
assert X_test_60_raw.shape == (248, 60, 15)


60-cycle tensors
----------------
Train: (37696, 60, 15) (37696,)
Val  : (8862, 60, 15) (8862,)
Test : (248, 60, 15) (248,)


In [ ]:
# ============================================================
# P2-E8.8 — SAVE WITHOUT OVERWRITING PAPER 1
# ============================================================

P2E8_ARTIFACT_PATH = OUTPUT_DIR / "P2_E8_60cycle_tensors.npz"

np.savez_compressed(
    P2E8_ARTIFACT_PATH,
    X_train=X_train_60_raw,
    y_train=y_train_60,
    X_val=X_val_60_raw,
    y_val=y_val_60,
    X_test=X_test_60_raw,
    y_test=y_test_60,
    train_engine_ids=np.asarray(p2e8_train_engine_ids),
    val_engine_ids=np.asarray(p2e8_val_engine_ids),
    train_endpoint_unit=train_units_60,
    train_endpoint_cycle=train_cycles_60,
    val_endpoint_unit=val_units_60,
    val_endpoint_cycle=val_cycles_60
)

print("Saved:", P2E8_ARTIFACT_PATH)


Saved: /content/drive/MyDrive/FD004_Paper2_Results/P2_E8_60cycle_tensors.npz


In [ ]:
# ============================================================
# P2-E8.9 — 14 SENSORS + 6 ONE-HOT REGIME CHANNELS
# ============================================================

def p2e8_make_onehot_inputs(X_raw):
    sensors = X_raw[:, :, :N_SENSORS].astype(np.float32)
    regimes = X_raw[:, :, N_SENSORS].astype(int)

    if regimes.min() < 0 or regimes.max() >= N_REGIMES:
        raise ValueError(
            f"Unexpected regime range: "
            f"{regimes.min()} to {regimes.max()}"
        )

    regime_oh = tf.keras.utils.to_categorical(
        regimes,
        num_classes=N_REGIMES
    ).astype(np.float32)

    return np.concatenate(
        [sensors, regime_oh],
        axis=-1
    )


X_train_60 = p2e8_make_onehot_inputs(X_train_60_raw)
X_val_60 = p2e8_make_onehot_inputs(X_val_60_raw)
X_test_60 = p2e8_make_onehot_inputs(X_test_60_raw)

print("Train:", X_train_60.shape)
print("Val  :", X_val_60.shape)
print("Test :", X_test_60.shape)

assert X_train_60.shape[1:] == (60, 20)
assert X_val_60.shape[1:] == (60, 20)
assert X_test_60.shape == (248, 60, 20)


Train: (37696, 60, 20)
Val  : (8862, 60, 20)
Test : (248, 60, 20)


In [ ]:
# ============================================================
# P2-E8.10 — CONTROLLED 60-CYCLE MODEL
# ============================================================

def build_p2e8_60cycle_model():
    inp = Input(
        shape=(60, 20),
        name="p2e8_60cycle_input"
    )

    x = Dense(
        PROJECTION_DIM,
        use_bias=False,
        name="linear_projection_18d"
    )(inp)

    x = LSTM(
        64,
        name="lstm_core"
    )(x)

    out = Dense(
        1,
        name="rul_head"
    )(x)

    model = Model(
        inp,
        out,
        name="P2_E8_60Cycle_Context"
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="mse",
        metrics=["mae"]
    )

    return model


set_all_seeds(SEED)
p2e8_model = build_p2e8_60cycle_model()
p2e8_model.summary()


Model: "P2_E8_60Cycle_Context"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ p2e8_60cycle_input (InputLayer) │ (None, 60, 20)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ linear_projection_18d (Dense)   │ (None, 60, 18)         │           360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_core (LSTM)                │ (None, 64)             │        21,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rul_head (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,673 (84.66 KB)

 Trainable params: 21,673 (84.66 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# P2-E8 training constants (defined locally so this section can be resumed safely)
EPOCHS = 60
BATCH_SIZE = 64
PATIENCE = 10


In [ ]:
# ============================================================
# P2-E8.11 — TRAIN
# ============================================================

callbacks_p2e8 = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=PATIENCE,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        OUTPUT_DIR / "P2_E8_60Cycle_Context.keras",
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=0
    )
]

history_p2e8 = p2e8_model.fit(
    X_train_60,
    y_train_60,
    validation_data=(X_val_60, y_val_60),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks_p2e8,
    verbose=1
)

print(
    "Training stopped after:",
    len(history_p2e8.history["loss"]),
    "epochs"
)


Epoch 1/60
589/589 ━━━━━━━━━━━━━━━━━━━━ 11s 10ms/step - loss: 5072.5840 - mae: 61.0819 - val_loss: 3235.2842 - val_mae: 49.0436
Epoch 2/60
589/589 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 2550.6340 - mae: 44.5589 - val_loss: 2030.1871 - val_mae: 40.4471
Epoch 3/60
589/589 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - loss: 1884.2046 - mae: 39.1888 - val_loss: 1802.6031 - val_mae: 38.1719
Epoch 4/60
589/589 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - loss: 1775.8187 - mae: 37.7713 - val_loss: 1788.0715 - val_mae: 37.7179
Epoch 5/60
589/589 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - loss: 1767.5090 - mae: 37.4807 - val_loss: 1789.4525 - val_mae: 37.6451
Epoch 6/60
589/589 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - loss: 1272.6787 - mae: 30.8728 - val_loss: 756.4511 - val_mae: 23.5624
Epoch 7/60
589/589 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 418.9637 - mae: 16.8542 - val_loss: 378.6191 - val_mae: 15.3140
Epoch 8/60
589/589 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - loss: 268.1788 - mae: 12.4717 - val_loss: 329.9382 - val_mae: 13

In [ ]:
def assign_life_stage(rul):
    rul = np.asarray(rul)
    return np.where(
        rul > 80, "Early",
        np.where(rul > 40, "Mid", "Late")
    )


def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    err = y_pred - y_true

    return {
        "N": len(y_true),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "Bias_pred_minus_true": float(np.mean(err))
    }


def evaluate_by_life_stage(y_true, y_pred):
    stages = assign_life_stage(y_true)
    rows = []

    for stage in ["Early", "Mid", "Late"]:
        mask = stages == stage
        if not np.any(mask):
            continue

        row = {"Life_Stage": stage}
        row.update(regression_metrics(y_true[mask], y_pred[mask]))
        rows.append(row)

    return pd.DataFrame(rows)


def evaluate_model(model, X, y, label="Model"):
    pred = model.predict(X, verbose=0).reshape(-1)

    global_row = {"Model": label}
    global_row.update(regression_metrics(y, pred))
    global_df = pd.DataFrame([global_row])

    stage_df = evaluate_by_life_stage(y, pred)
    stage_df.insert(0, "Model", label)

    return pred, global_df, stage_df

In [ ]:
# ============================================================
# P2-E8.12 — STANDARD VALIDATION METRICS
# ============================================================

pred_p2e8, global_p2e8, stage_p2e8 = evaluate_model(
    p2e8_model,
    X_val_60,
    y_val_60,
    label="P2-E8 60-Cycle Context"
)

print("\n" + "="*60)
print("P2-E8 GLOBAL VALIDATION PERFORMANCE")
print("="*60)
display(global_p2e8)

print("\n" + "="*60)
print("P2-E8 PERFORMANCE BY LIFE STAGE")
print("="*60)
display(stage_p2e8)



P2-E8 GLOBAL VALIDATION PERFORMANCE


,Model,N,RMSE,MAE,Bias_pred_minus_true
0,P2-E8 60-Cycle Context,8862,17.431611,11.652758,-1.01507



P2-E8 PERFORMANCE BY LIFE STAGE


,Model,Life_Stage,N,RMSE,MAE,Bias_pred_minus_true
0,P2-E8 60-Cycle Context,Early,4908,17.462558,11.465833,-5.350217
1,P2-E8 60-Cycle Context,Mid,1945,23.788393,18.594591,6.510595
2,P2-E8 60-Cycle Context,Late,2009,6.895188,5.388722,2.289800


In [ ]:
# ============================================================
# P2-E8.13 — PRELIMINARY GATE
# Final decision requires matched endpoints.
# ============================================================

BASELINE_GLOBAL_RMSE = 18.250101
BASELINE_MID_RMSE = 27.739052
BEST_LEAKAGE_SAFE_MID_RMSE = 26.091819

p2e8_global_rmse = float(global_p2e8["RMSE"].iloc[0])
p2e8_mid_rmse = float(
    stage_p2e8.loc[
        stage_p2e8["Life_Stage"] == "Mid",
        "RMSE"
    ].iloc[0]
)

print("Global RMSE:", p2e8_global_rmse)
print("Mid RMSE   :", p2e8_mid_rmse)
print()
print("Global <= P2-E1:",
      p2e8_global_rmse <= BASELINE_GLOBAL_RMSE)
print("Mid < P2-E1:",
      p2e8_mid_rmse < BASELINE_MID_RMSE)
print("Mid < P2-E5:",
      p2e8_mid_rmse < BEST_LEAKAGE_SAFE_MID_RMSE)

print(
    "\nPreliminary only: final P2-E8 decision requires "
    "matched 30/60-cycle validation endpoints."
)


Global RMSE: 17.431610753650876
Mid RMSE   : 23.78839284771764

Global <= P2-E1: True
Mid < P2-E1: True
Mid < P2-E5: True

Preliminary only: final P2-E8 decision requires matched 30/60-cycle validation endpoints.


## P2-E8 matched-endpoint gate — required after training

Because a 60-cycle window removes early endpoints that are present in the 30-cycle validation set, the ordinary P2-E8 table is not by itself a fair head-to-head comparison.

After training:

1. regenerate 30-cycle validation windows with `(unit_id, time_cycles)` metadata from the same reconstructed validation engines;
2. keep only 30-cycle endpoints also present in the 60-cycle endpoint set;
3. evaluate the **trained best-checkpoint** P2-E1 model on those matched 30-cycle windows (never a freshly initialized model);
4. compare P2-E1 and P2-E8 on identical targets;
5. report matched global and Early/Mid/Late RMSE.

Only this matched comparison should determine whether the longer context is promoted.

If P2-E8 fails the matched gate, stop adding more handcrafted 30-cycle features and move to the physics-informed / degradation-constrained branch.


In [ ]:
# ============================================================
# P2-E8.14 — MATCHED-ENDPOINT 30-vs-60 FAIR COMPARISON
# ============================================================
# Rebuild 30-cycle validation windows WITH endpoint metadata.
(
    X_val_30_match_raw,
    y_val_30_match_all,
    val_units_30,
    val_cycles_30
) = p2e8_generate_windows_train_val(
    p2e8_val_split_df,
    30,
    P2E8_FEATURE_COLS,
    return_metadata=True
)

# Endpoint key -> 30-cycle row. Endpoints are unique within unit.
idx30 = {
    (int(u), int(c)): i
    for i, (u, c) in enumerate(zip(val_units_30, val_cycles_30))
}
keys60 = [
    (int(u), int(c))
    for u, c in zip(val_units_60, val_cycles_60)
]
missing = [k for k in keys60 if k not in idx30]
assert not missing, f"Matched-endpoint failure: {len(missing)} 60-cycle endpoints absent from 30-cycle set"

rows30 = np.asarray([idx30[k] for k in keys60], dtype=int)
X_val_30_matched_raw = X_val_30_match_raw[rows30]
y_val_30_matched = y_val_30_match_all[rows30]

# Exact same targets/order must be used for the candidate and baseline.
np.testing.assert_allclose(y_val_30_matched, y_val_60, rtol=0, atol=1e-6)

X_val_30_matched = p2e8_make_onehot_inputs(X_val_30_matched_raw)

# Use the trained frozen/reproduced P2-E1 control. Prefer the saved best checkpoint
# so this gate cannot accidentally use a freshly initialized baseline_model.
p2e1_ckpt = OUTPUT_DIR / "P2_E1_baseline.keras"
if p2e1_ckpt.exists():
    p2e1_matched_model = tf.keras.models.load_model(p2e1_ckpt)
    print("Loaded frozen P2-E1 checkpoint:", p2e1_ckpt)
elif 'history_baseline' in globals() and 'baseline_model' in globals():
    p2e1_matched_model = baseline_model
    print("Using in-memory trained P2-E1 baseline_model.")
else:
    raise RuntimeError(
        "No trained P2-E1 control is available. Run P2-E1 training first or place "
        "P2_E1_baseline.keras in OUTPUT_DIR before the matched-endpoint gate."
    )

pred30 = p2e1_matched_model.predict(X_val_30_matched, verbose=0).reshape(-1)
pred60 = p2e8_model.predict(X_val_60, verbose=0).reshape(-1)

global30 = regression_metrics(y_val_60, pred30)
global60 = regression_metrics(y_val_60, pred60)
stage30 = evaluate_by_life_stage(y_val_60, pred30)
stage60 = evaluate_by_life_stage(y_val_60, pred60)

matched_global = pd.DataFrame([
    {"Model": "P2-E1 30-cycle matched", **global30},
    {"Model": "P2-E8 60-cycle matched", **global60},
])

stage30 = stage30.copy(); stage30["Model"] = "P2-E1 30-cycle matched"
stage60 = stage60.copy(); stage60["Model"] = "P2-E8 60-cycle matched"
matched_stage = pd.concat([stage30, stage60], ignore_index=True)

print("Matched endpoints:", len(y_val_60))
print("\nGLOBAL MATCHED COMPARISON")
display(matched_global)
print("\nLIFE-STAGE MATCHED COMPARISON")
display(matched_stage)

rmse30 = float(matched_global.loc[matched_global.Model == "P2-E1 30-cycle matched", "RMSE"].iloc[0])
rmse60 = float(matched_global.loc[matched_global.Model == "P2-E8 60-cycle matched", "RMSE"].iloc[0])
mid30 = float(matched_stage.loc[(matched_stage.Model == "P2-E1 30-cycle matched") & (matched_stage.Life_Stage == "Mid"), "RMSE"].iloc[0])
mid60 = float(matched_stage.loc[(matched_stage.Model == "P2-E8 60-cycle matched") & (matched_stage.Life_Stage == "Mid"), "RMSE"].iloc[0])

print("\nP2-E8 MATCHED GATE")
print(f"Global RMSE: 30={rmse30:.4f} | 60={rmse60:.4f} | delta={rmse60-rmse30:+.4f}")
print(f"Mid RMSE   : 30={mid30:.4f} | 60={mid60:.4f} | delta={mid60-mid30:+.4f}")
print("Global improved:", rmse60 < rmse30)
print("Mid-life improved:", mid60 < mid30)

matched_global.to_csv(OUTPUT_DIR / "P2_E8_matched_global_comparison.csv", index=False)
matched_stage.to_csv(OUTPUT_DIR / "P2_E8_matched_stage_comparison.csv", index=False)
print("Saved matched comparison tables to:", OUTPUT_DIR)


Loaded frozen P2-E1 checkpoint: /content/drive/MyDrive/FD004_Paper2_Results/P2_E1_baseline.keras
Matched endpoints: 8862

GLOBAL MATCHED COMPARISON


,Model,N,RMSE,MAE,Bias_pred_minus_true
0,P2-E1 30-cycle matched,8862,19.116306,13.378009,0.163992
1,P2-E8 60-cycle matched,8862,17.431611,11.652758,-1.015070



LIFE-STAGE MATCHED COMPARISON


,Life_Stage,N,RMSE,MAE,Bias_pred_minus_true,Model
0,Early,4908,17.537906,12.431658,-6.447741,P2-E1 30-cycle matched
1,Mid,1945,27.704070,21.917965,12.730473,P2-E1 30-cycle matched
2,Late,2009,10.839741,7.422046,4.150345,P2-E1 30-cycle matched
3,Early,4908,17.462558,11.465833,-5.350217,P2-E8 60-cycle matched
4,Mid,1945,23.788393,18.594591,6.510595,P2-E8 60-cycle matched
5,Late,2009,6.895188,5.388722,2.289800,P2-E8 60-cycle matched



P2-E8 MATCHED GATE
Global RMSE: 30=19.1163 | 60=17.4316 | delta=-1.6847
Mid RMSE   : 30=27.7041 | 60=23.7884 | delta=-3.9157
Global improved: True
Mid-life improved: True
Saved matched comparison tables to: /content/drive/MyDrive/FD004_Paper2_Results


## P2-E8 final IID/development decision — PASS

The mandatory matched-endpoint comparison used **8,862 identical validation targets** for both models.

| Metric | P2-E1 30-cycle matched | P2-E8 60-cycle matched | Change |
|---|---:|---:|---:|
| Global RMSE | 19.1163 | **17.4316** | **-1.6847** |
| Mid-life RMSE | 27.7041 | **23.7884** | **-3.9157** |
| Early-life RMSE | 17.5379 | **17.4626** | -0.0753 |
| Late-life RMSE | 10.8397 | **6.8952** | **-3.9446** |

**Decision:** P2-E8 passes the controlled IID/development gate because both global and mid-life RMSE improve on identical endpoints. Freeze this candidate before Shift-V1 / Shift-V2 evaluation; do not tune it using DS-A or DS-B outcomes.


# Validation execution status — completed

The planned sequence has now been executed:

1. ✅ **Shift-V1 / DS-A** — P2-E8 prediction export created and validated.
2. ✅ **Shift-V2 / DS-B** — P2-E8 prediction export created and validated.
3. ✅ **Shift-V3** — persistent nine-engine diagnostics completed.
4. ✅ **UQ-V1 / DS-A** — Asymmetric and Balanced Engine CQR evaluated.
5. ✅ **UQ-V1 / DS-B** — Asymmetric and Balanced Engine CQR evaluated.

No further tuning should be performed using these frozen shift-test outcomes.

### Remaining project actions

1. Freeze the final result package: notebooks, checkpoints, prediction CSVs, UQ summaries, engine-level metrics, configuration, and seed.
2. Create final Paper 1 vs Paper 2 tables/figures.
3. Optionally run a **multi-seed confirmation** of the already frozen P2-E8 specification as a robustness check; do not use it to redesign the model.
4. Write the Results, Discussion, Limitations, and Conclusion sections.


# 11. Future validation gate Shift-V1 — DS-A held-out operating-regime evaluation

**Do not run this as a tuning loop.** Run it only after an IID/development candidate has been selected and frozen.

Use the frozen DS-A protocol from Paper 1:

- Regime 0 held out from training/calibration prediction endpoints
- evaluation on held-out-regime endpoints from unseen test engines
- engine-level separation preserved
- complete consecutive 30-cycle histories preserved

### Paper 1 reference

- Point RMSE: **24.869**
- Point MAE: **14.869**
- Balanced Engine CQR coverage: **90.21%**
- Mid-life Balanced coverage: **70.88%**

### Validation question

Does the selected Paper 2 point model reduce overall and mid-life error under the held-out-regime shift without having been tuned on DS-A test outcomes?

In [ ]:
# ============================================================
# SHIFT-V1 — DS-A EXPORT CHECK / NEXT ACTION
# ============================================================
DS_A_REQUIRED_COLUMNS = ["unit_id", "cycle", "RUL", "prediction", "regime"]
DS_A_EXPORT = OUTPUT_DIR / "DS_A_P2E8_predictions.csv"

print("Shift-V1 / DS-A")
print("Expected export:", DS_A_EXPORT)
print("Required columns:", DS_A_REQUIRED_COLUMNS)

if DS_A_EXPORT.exists():
    ds_a_p2e8 = pd.read_csv(DS_A_EXPORT)
    missing = set(DS_A_REQUIRED_COLUMNS) - set(ds_a_p2e8.columns)
    if missing:
        raise ValueError(f"DS-A export is missing columns: {sorted(missing)}")
    print(f"PASS: loaded DS-A P2-E8 predictions ({len(ds_a_p2e8):,} rows).")
    display(ds_a_p2e8.head())
else:
    print("\nNEXT ACTION:")
    print("1. Open the frozen Paper 1 DS-A notebook: 03_FD004_DS_A_Heldout_Regime (1).ipynb")
    print("2. Preserve its held-out-regime split and engine separation.")
    print("3. Replace only the point predictor with the frozen P2-E8 60-cycle model/checkpoint.")
    print("4. Export endpoint predictions to the path shown above.")
    print("5. Return here and rerun this cell.")


Shift-V1 / DS-A
Expected export: /content/drive/MyDrive/FD004_Paper2_Results/DS_A_P2E8_predictions.csv
Required columns: ['unit_id', 'cycle', 'RUL', 'prediction', 'regime']
PASS: loaded DS-A P2-E8 predictions (1,753 rows).


,unit_id,cycle,RUL,prediction,regime
0,17,64,87.0,122.615669,0
1,17,69,82.0,120.370583,0
2,17,82,69.0,115.488243,0
3,17,83,68.0,111.865723,0
4,17,87,64.0,103.897652,0


# 12. Future validation gate Shift-V2 — DS-B restricted operating-envelope evaluation

Run only after the Paper 2 candidate is selected and frozen.

Use the frozen DS-B protocol from Paper 1:

- all six regimes remain present
- training/calibration endpoints are inside the regime-specific central operating envelope
- test endpoints are outside the envelope
- engine separation and 30-cycle consecutive histories are preserved

### Paper 1 reference

- Point RMSE: **23.999**
- Point MAE: **15.329**
- Balanced Engine CQR coverage: **90.63%**
- Mid-life Balanced coverage: **75.16%**

### Validation question

Does the selected Paper 2 model improve a second, distinct operating-condition shift without overfitting the DS-A mechanism?

In [ ]:
# ============================================================
# SHIFT-V2 — DS-B EXPORT CHECK / NEXT ACTION
# ============================================================
DS_B_REQUIRED_COLUMNS = [
    "unit_id", "cycle", "RUL", "prediction", "regime",
    "envelope_score", "envelope_threshold"
]
DS_B_EXPORT = OUTPUT_DIR / "DS_B_P2E8_predictions.csv"

print("Shift-V2 / DS-B")
print("Expected export:", DS_B_EXPORT)
print("Required columns:", DS_B_REQUIRED_COLUMNS)

if DS_B_EXPORT.exists():
    ds_b_p2e8 = pd.read_csv(DS_B_EXPORT)
    missing = set(DS_B_REQUIRED_COLUMNS) - set(ds_b_p2e8.columns)
    if missing:
        raise ValueError(f"DS-B export is missing columns: {sorted(missing)}")
    print(f"PASS: loaded DS-B P2-E8 predictions ({len(ds_b_p2e8):,} rows).")
    display(ds_b_p2e8.head())
else:
    print("\nNEXT ACTION:")
    print("1. Open the frozen Paper 1 DS-B restricted-envelope notebook.")
    print("2. Preserve its regime-specific operating-envelope split and engine separation.")
    print("3. Replace only the point predictor with the frozen P2-E8 60-cycle model/checkpoint.")
    print("4. Export endpoint predictions to the path shown above.")
    print("5. Return here and rerun this cell.")


Shift-V2 / DS-B
Expected export: /content/drive/MyDrive/FD004_Paper2_Results/DS_B_P2E8_predictions.csv
Required columns: ['unit_id', 'cycle', 'RUL', 'prediction', 'regime', 'envelope_score', 'envelope_threshold']
PASS: loaded DS-B P2-E8 predictions (1,408 rows).


,unit_id,cycle,RUL,prediction,regime,envelope_score,envelope_threshold
0,17,70,81.0,122.979836,1,1.371685,1.006563
1,17,71,80.0,121.545837,4,1.091058,1.018319
2,17,73,78.0,122.239159,2,1.354804,1.009493
3,17,78,73.0,122.117088,1,1.126035,1.006563
4,17,81,70.0,119.906311,2,1.092483,1.009493


# 13. Shift-V3 — Persistent shift-sensitive engine analysis — COMPLETED

Paper 1 identified nine engines below the 80% Balanced-CQR target under both DS-A and DS-B:

**17, 105, 120, 156, 181, 184, 194, 240, 245**

These engines remain a **diagnostic subgroup, not a tuning set**.

### P2-E8 point-error result

| Shift | Group | Scope | N | RMSE | MAE | Bias (pred − true) |
|---|---|---|---:|---:|---:|---:|
| DS-A | Persistent | All | 271 | **24.881** | 19.609 | +7.015 |
| DS-A | Persistent | Mid | 95 | **31.952** | 28.258 | **+15.769** |
| DS-A | Other | All | 1482 | 15.044 | 10.982 | -2.997 |
| DS-A | Other | Mid | 290 | 17.856 | 13.838 | +5.631 |
| DS-B | Persistent | All | 234 | **26.644** | 20.351 | +6.716 |
| DS-B | Persistent | Mid | 89 | **32.350** | 27.144 | **+15.875** |
| DS-B | Other | All | 1174 | 14.506 | 10.606 | -1.986 |
| DS-B | Other | Mid | 226 | 19.245 | 15.586 | +6.173 |

### Decision

**PARTIAL PASS.** P2-E8 substantially improves broad DS-A/DS-B point accuracy, but the persistent subgroup remains disproportionately difficult. The main residual failure mode is mid-life overprediction for these engines. Do not retune P2-E8 on this subgroup; retain it as a limitation and diagnostic target for future work.


In [ ]:
# ============================================================
# SHIFT-V3 — EXECUTABLE PERSISTENT-ENGINE DIAGNOSTICS
# Requires DS_A_P2E8_predictions.csv and/or DS_B_P2E8_predictions.csv
# ============================================================
def assign_life_stage(rul):
    rul = np.asarray(rul)
    return np.where(
        rul > 80, "Early",
        np.where(rul > 40, "Mid", "Late")
    )

def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    err = y_pred - y_true

    return {
        "N": len(y_true),
        "RMSE": float(np.sqrt(np.mean((y_pred - y_true) ** 2))),
        "MAE": float(np.mean(np.abs(y_pred - y_true))),
        "Bias_pred_minus_true": float(np.mean(err))
    }

def persistent_engine_diagnostics(prediction_df, shift_label="Shift"):
    required = {"unit_id", "RUL", "prediction"}
    missing = required - set(prediction_df.columns)
    if missing:
        raise ValueError(f"{shift_label}: missing required columns: {sorted(missing)}")

    df = prediction_df.copy()
    df["Persistent_Problem_Engine"] = df["unit_id"].isin(PERSISTENT_SHIFT_ENGINES)
    df["Life_Stage"] = assign_life_stage(df["RUL"].to_numpy())

    rows = []
    for group_name, mask in [
        ("Persistent", df["Persistent_Problem_Engine"]),
        ("Other", ~df["Persistent_Problem_Engine"]),
    ]:
        if mask.sum() == 0:
            continue
        m = regression_metrics(df.loc[mask, "RUL"], df.loc[mask, "prediction"])
        m.update({"Shift": shift_label, "Group": group_name, "Life_Stage": "All"})
        rows.append(m)

        mid_mask = mask & (df["Life_Stage"] == "Mid")
        if mid_mask.sum():
            m = regression_metrics(df.loc[mid_mask, "RUL"], df.loc[mid_mask, "prediction"])
            m.update({"Shift": shift_label, "Group": group_name, "Life_Stage": "Mid"})
            rows.append(m)

    return pd.DataFrame(rows)[[
        "Shift", "Group", "Life_Stage", "N", "RMSE", "MAE", "Bias_pred_minus_true"
    ]]

shift_tables = []
available = []

for label, path in [
    ("DS-A", OUTPUT_DIR / "DS_A_P2E8_predictions.csv"),
    ("DS-B", OUTPUT_DIR / "DS_B_P2E8_predictions.csv"),
]:
    if path.exists():
        df_shift = pd.read_csv(path)
        shift_tables.append(persistent_engine_diagnostics(df_shift, label))
        available.append(label)
    else:
        print(f"{label}: prediction export not found -> {path}")

if shift_tables:
    shift_v3_metrics = pd.concat(shift_tables, ignore_index=True)
    print("\nSHIFT-V3 PERSISTENT-ENGINE DIAGNOSTICS")
    display(shift_v3_metrics)

    out_path = OUTPUT_DIR / "Shift_V3_persistent_engine_metrics.csv"
    shift_v3_metrics.to_csv(out_path, index=False)
    print("Saved:", out_path)
    print("Completed shifts:", ", ".join(available))
else:
    print("\nSHIFT-V3 NOT READY YET.")
    print("Generate the DS-A and DS-B P2-E8 prediction exports in Sections 11 and 12 first.")
    print("Then rerun this cell; no model retraining is required here.")



SHIFT-V3 PERSISTENT-ENGINE DIAGNOSTICS


,Shift,Group,Life_Stage,N,RMSE,MAE,Bias_pred_minus_true
0,DS-A,Persistent,All,271,24.881099,19.609134,7.014937
1,DS-A,Persistent,Mid,95,31.951898,28.258225,15.768669
2,DS-A,Other,All,1482,15.044259,10.982008,-2.997339
3,DS-A,Other,Mid,290,17.855582,13.837589,5.630870
4,DS-B,Persistent,All,234,26.643890,20.350808,6.716277
5,DS-B,Persistent,Mid,89,32.350335,27.143761,15.874852
6,DS-B,Other,All,1174,14.506319,10.606442,-1.986092
7,DS-B,Other,Mid,226,19.245424,15.586044,6.173319


Saved: /content/drive/MyDrive/FD004_Paper2_Results/Shift_V3_persistent_engine_metrics.csv
Completed shifts: DS-A, DS-B


## Project progress checkpoint — primary experiments complete

- **Paper 1 diagnosis:** distribution-shift reliability failures were strongly associated with point-prediction error, particularly during mid-life degradation.
- **P2-E1 to P2-E7:** established the control, oracle mechanism, and negative/partial leakage-safe engineered-feature results.
- **P2-E8:** 60-cycle context is the first leakage-safe candidate to pass the fair IID matched-endpoint gate.
- **Shift-V1 / DS-A:** dedicated point RMSE **16.942**, substantially below the Paper 1 reference **24.869**.
- **Shift-V2 / DS-B:** dedicated point RMSE **17.130**, substantially below the Paper 1 reference **23.999**.
- **Shift-V3:** persistent engines remain a concentrated failure subgroup, especially at mid-life.
- **UQ-V1:** Balanced Engine CQR remains more reliable than Asymmetric CQR. Mid-life coverage improves strongly, while persistent-engine and worst-engine reliability remain incomplete.

### Project phase now

The project is no longer in architecture search. It is in **freeze, consolidation, reproducibility checking, figure/table generation, and manuscript preparation**.


# 14. UQ-V1 — Conformal uncertainty validation — COMPLETED

The frozen Paper 1 uncertainty pipeline was reapplied using a **60-cycle P2-E8-style quantile backbone** with q10/q50/q90 outputs, followed by:

1. **Asymmetric CQR**
2. **Balanced Engine CQR**

The original DS-A/DS-B split logic and untouched conformal calibration-engine protocol were preserved.

## UQ-V1 results

| Shift | Method | Global coverage | Mean width | Mid-life coverage | Persistent-9 coverage | Persistent-9 mid coverage |
|---|---|---:|---:|---:|---:|---:|
| DS-A | Asymmetric CQR | 69.709% | 26.385 | 66.753% | 54.613% | 45.263% |
| DS-A | **Balanced Engine CQR** | **85.054%** | 45.342 | **81.558%** | **70.849%** | **58.947%** |
| DS-B | Asymmetric CQR | 76.918% | 34.365 | 65.714% | 55.556% | 51.685% |
| DS-B | **Balanced Engine CQR** | **89.205%** | 56.887 | **86.349%** | **78.205%** | **75.281%** |

## Engine-level reliability

| Shift | Asymmetric engines ≥80% | Balanced engines ≥80% | Asymmetric worst engine | Balanced worst engine |
|---|---:|---:|---:|---:|
| DS-A | 10 / 38 | **26 / 38** | 8.824% | **44.118%** |
| DS-B | 17 / 38 | **33 / 38** | 18.182% | **54.545%** |

## Paper 1 vs Paper 2 Balanced-CQR interpretation

Paper 1 Balanced-CQR global coverage was **90.21% on DS-A** and **90.63% on DS-B**. Paper 2 gives **85.054%** and **89.205%**, respectively, so global coverage does not uniformly improve.

The important life-stage result is different: Paper 1 mid-life Balanced coverage was **70.88% on DS-A** and **75.16% on DS-B**, while Paper 2 reaches **81.558%** and **86.349%**. Thus mid-life coverage improves by approximately **+10.68 percentage points on DS-A** and **+11.19 points on DS-B**.

### Methodological note — do not mix the two point-RMSE quantities

The dedicated single-output P2-E8 point-model shift notebooks achieved **16.942 RMSE (DS-A)** and **17.130 RMSE (DS-B)**. The UQ notebooks train a separate three-quantile model for CQR; its median/point-like RMSE is **18.388 (DS-A)** and **18.997 (DS-B)**. These values answer different questions and should not be presented as if they came from the same fitted model.

### UQ-V1 decision

**Balanced Engine CQR remains the preferred uncertainty method.** UQ-V1 is a **partial overall success**: the targeted mid-life reliability failure improves substantially, especially on DS-B, but global coverage and persistent-engine/worst-engine reliability remain incomplete. This residual subgroup behavior should be reported as a limitation rather than used for further tuning.


In [5]:
# ============================================================
# UQ-V1 CONSOLIDATION CHECK
# Loads the saved DS-A / DS-B UQ summaries if they are available.
# No training is performed here.
# ============================================================
from pathlib import Path
import pandas as pd

UQ_A_PATH = Path("/content/drive/MyDrive/FD004_Paper2_Results/UQ_V1_DS_A/UQ_V1_DS_A_summary.csv")
UQ_B_PATH = Path("/content/drive/MyDrive/FD004_Paper2_Results/UQ_V1_DS_B/UQ_V1_DS_B_summary.csv")

uq_tables = []

for shift, path in [("DS-A", UQ_A_PATH), ("DS-B", UQ_B_PATH)]:
    if path.exists():
        df = pd.read_csv(path)
        uq_tables.append(df)
        print(f"PASS: loaded {shift} UQ summary -> {path}")
    else:
        print(f"NOTE: {shift} UQ summary not found at {path}")

if uq_tables:
    uq_v1_consolidated = pd.concat(uq_tables, ignore_index=True)
    display(uq_v1_consolidated)
else:
    print("UQ summaries were already reviewed in the executed UQ notebooks.")
    print("Copy/save their CSV exports to the paths above if you want this cell to display them here.")


PASS: loaded DS-A UQ summary -> /content/drive/MyDrive/FD004_Paper2_Results/UQ_V1_DS_A/UQ_V1_DS_A_summary.csv
PASS: loaded DS-B UQ summary -> /content/drive/MyDrive/FD004_Paper2_Results/UQ_V1_DS_B/UQ_V1_DS_B_summary.csv


,Shift,Method,Group,Life_Stage,N,Coverage,Mean_Width
0,DS-A,Asymmetric CQR,All,All,1753,69.709070,26.384571
1,DS-A,Asymmetric CQR,All,Early,967,64.012410,25.819695
2,DS-A,Asymmetric CQR,All,Mid,385,66.753247,31.647877
3,DS-A,Asymmetric CQR,All,Late,401,86.284289,22.693455
4,DS-A,Asymmetric CQR,Persistent9,All,271,54.612546,28.401781
5,DS-A,Asymmetric CQR,Persistent9,Mid,95,45.263158,33.536289
6,DS-A,Balanced Engine CQR,All,All,1753,85.054193,45.342449
7,DS-A,Balanced Engine CQR,All,Early,967,81.282316,45.636482
8,DS-A,Balanced Engine CQR,All,Mid,385,81.558442,51.266945
9,DS-A,Balanced Engine CQR,All,Late,401,97.506234,38.945267


# 15. Master model-comparison table

Maintain one auditable master table throughout Paper 2.

Do not overwrite earlier experiments. Every reported result should remain traceable to a saved notebook/checkpoint, configuration, prediction export, and seed.

**Important:** keep dedicated single-output point-model results separate from the quantile-model median results used by UQ-V1.


In [6]:
PAPER2_COLUMNS = [
    "Experiment",
    "Model",
    "Seed",
    "Condition",
    "Global_RMSE",
    "Global_MAE",
    "Global_Bias",
    "Early_RMSE",
    "Mid_RMSE",
    "Late_RMSE",
    "Persistent_Engine_RMSE",
    "Parameter_Count",
    "Notes"
]

paper2_master = pd.DataFrame(columns=PAPER2_COLUMNS)
paper2_master

,Experiment,Model,Seed,Condition,Global_RMSE,Global_MAE,Global_Bias,Early_RMSE,Mid_RMSE,Late_RMSE,Persistent_Engine_RMSE,Parameter_Count,Notes


# 16. Final figures recommended for Paper 2

Prepare the following manuscript figures from the frozen outputs:

1. **Paper 1 vs P2-E8 point RMSE** for IID matched endpoints, DS-A, and DS-B.
2. **Early/Mid/Late RMSE** comparison, emphasizing the mid-life reduction.
3. **Persistent vs other engines** under DS-A and DS-B.
4. **Asymmetric vs Balanced Engine CQR** coverage and mean width.
5. **Balanced coverage by life stage** for Paper 1 vs Paper 2.
6. **Per-engine coverage distribution**, with the 80% reliability threshold.
7. Optional representative RUL trajectories for persistent and non-persistent engines.

Use the frozen CSV/checkpoint outputs only; do not regenerate results with changed hyperparameters for figure production.


In [7]:
def plot_stage_rmse_comparison(stage_tables):
    # stage_tables: dict {"Model Name": dataframe returned by evaluate_by_life_stage}
    rows = []

    for model_name, df in stage_tables.items():
        temp = df[["Life_Stage", "RMSE"]].copy()
        temp["Model"] = model_name
        rows.append(temp)

    plot_df = pd.concat(rows, ignore_index=True)
    pivot = plot_df.pivot(index="Life_Stage", columns="Model", values="RMSE")
    pivot = pivot.reindex(["Early", "Mid", "Late"])

    ax = pivot.plot(kind="bar", figsize=(9, 5))
    ax.set_ylabel("RMSE (cycles)")
    ax.set_xlabel("Life Stage")
    ax.set_title("Paper 2: Point-Model RMSE by Life Stage")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

# 17. Final Paper 2 experimental results — consolidated summary

## A. IID/development results

| Experiment | Model | Global RMSE | Early RMSE | Mid RMSE | Late RMSE | Decision |
|---|---|---:|---:|---:|---:|---|
| **P2-E1** | Frozen Paper 1 baseline | 18.250 | 16.331 | 27.739 | 10.840 | PASS / control |
| **P2-E2** | Oracle stage | 10.857 | 12.009 | 11.181 | 5.250 | PASS diagnostic only |
| **P2-E3** | Predicted stage probabilities | 20.100 | 18.597 | 28.744 | 13.274 | FAIL |
| **P2-E4** | Continuous degradation auxiliary target | 19.031 | 17.074 | 28.267 | 12.640 | FAIL |
| **P2-E5** | Sensor trend / slope aware | 19.860 | 19.613 | 26.092 | 12.077 | PARTIAL |
| **P2-E6** | Residual trend correction | 18.332 | 17.154 | 26.659 | 10.143 | PARTIAL / not promoted |
| **P2-E6b** | OOF residual correction | 18.229 | 16.238 | 27.726 | 11.126 | FAIL / not promoted |
| **P2-E7** | Delta / first-difference aware | 19.876 | 16.882 | 31.118 | 13.578 | FAIL |
| **P2-E8** | 60-cycle context, matched endpoints | **17.432** | **17.463** | **23.788** | **6.895** | **PASS / promoted** |

P2-E8 reduces matched global RMSE by **8.8%** and matched mid-life RMSE by **14.1%** relative to the matched P2-E1 control.

## B. Frozen distribution-shift point validation

| Shift | Paper 1 point RMSE | P2-E8 point RMSE | RMSE reduction | Decision |
|---|---:|---:|---:|---|
| **DS-A** | 24.869 | **16.942** | **31.9%** | **PASS** |
| **DS-B** | 23.999 | **17.130** | **28.6%** | **PASS** |

This is the strongest deployable Paper 2 result: the longer-context model generalizes across two distinct operating-condition shifts without being tuned on either shift test set.

## C. Persistent-engine diagnostic

The global shift improvement is not uniform. The persistent nine-engine subgroup retains approximately **32-cycle mid-life RMSE** under both DS-A and DS-B and a systematic positive mid-life bias of about **+15.8 cycles**. This subgroup remains the main point-prediction limitation.

## D. UQ-V1

Balanced Engine CQR clearly dominates Asymmetric CQR in coverage stability. Relative to Paper 1, **mid-life Balanced coverage improves materially**:

- DS-A: **70.88% → 81.558%**
- DS-B: **75.16% → 86.349%**

However, global Balanced coverage is **85.054% on DS-A** and **89.205% on DS-B**, compared with Paper 1 references of **90.21%** and **90.63%**. Persistent-engine and worst-engine coverage also remain below the desired reliability level.

## Overall Paper 2 conclusion

> **The central deployable finding is that broader temporal context, rather than handcrafted degradation-state features, substantially reduces the mid-life RUL error that drove Paper 1 distribution-shift failures. The 60-cycle P2-E8 model improves point accuracy under IID, DS-A, and DS-B. These point improvements translate into substantially better mid-life conformal coverage, but do not completely eliminate engine-specific uncertainty failures.**

This supports the Paper 2 hypothesis in a refined form: degradation information matters, but the successful observable mechanism is **longer trajectory context** rather than explicit inferred stage labels or simple derivative features.


# 18. Final stopping / promotion decision

### P2-E8 promotion decision

P2-E8 satisfies the main promotion requirements used in this project:

- ✅ meaningful mid-life RMSE reduction on matched IID endpoints,
- ✅ no unacceptable broad performance degradation,
- ✅ leakage-safe 60-cycle reconstruction passed the frozen Paper 1 integrity gate,
- ✅ substantial point-RMSE improvement under **both DS-A and DS-B**,
- ✅ architecture was selected before observing the frozen shift-test outcomes.

The persistent-engine subgroup does **not** fully improve, so this is retained as a limitation rather than hidden through additional test-driven tuning.

### Final experimental stopping rule

Stop architecture search at P2-E8 for this paper. Additional model changes after seeing DS-A/DS-B/UQ results would contaminate the clean validation story.

A multi-seed rerun of the **unchanged frozen P2-E8 specification** may still be performed as a reproducibility check before submission. It should confirm stability, not trigger a new tuning loop.


# 19. Final project checklist

## Completed

1. ✅ P2-E1 frozen baseline.
2. ✅ P2-E2 oracle-stage diagnostic.
3. ✅ P2-E3 predicted-stage model.
4. ✅ P2-E4 continuous auxiliary degradation model.
5. ✅ P2-E5 trend/slope-aware model.
6. ✅ P2-E6 residual correction.
7. ✅ P2-E6b OOF residual correction.
8. ✅ P2-E7 first-difference model.
9. ✅ P2-E8 30-cycle reconstruction/integrity gate.
10. ✅ P2-E8 60-cycle training with early stopping.
11. ✅ Matched 30-vs-60 endpoint comparison.
12. ✅ Shift-V1 / DS-A point validation.
13. ✅ Shift-V2 / DS-B point validation.
14. ✅ Shift-V3 persistent-engine diagnostics.
15. ✅ UQ-V1 / DS-A.
16. ✅ UQ-V1 / DS-B.

## Remaining before final manuscript freeze

17. ⬜ Archive/copy all checkpoints, prediction CSVs, UQ summaries, engine metrics, and configuration files into one immutable Paper 2 results package.
18. ⬜ Produce the final Paper 1 vs Paper 2 tables and figures.
19. ⬜ Optional but recommended: rerun the **unchanged P2-E8 specification across additional seeds** to document variability.
20. ⬜ Write Results and Discussion, explicitly separating:
   - dedicated point-model RMSE,
   - quantile-model median RMSE used in UQ,
   - conformal coverage/width.
21. ⬜ Write limitations: persistent-engine mid-life overprediction and residual worst-engine undercoverage.
22. ⬜ Freeze the final notebook/report package and record file hashes/version identifiers if required by the research workflow.


In [8]:
# ============================================================
# FINAL PAPER 2 FREEZE SUMMARY
# This cell records the headline results; it does not retrain anything.
# ============================================================
final_paper2_summary = pd.DataFrame([
    {"Stage": "IID matched", "Metric": "Global RMSE", "Paper1_or_P2E1": 19.1163, "Paper2_P2E8": 17.4316},
    {"Stage": "IID matched", "Metric": "Mid-life RMSE", "Paper1_or_P2E1": 27.7041, "Paper2_P2E8": 23.7884},
    {"Stage": "DS-A point", "Metric": "Global RMSE", "Paper1_or_P2E1": 24.8690, "Paper2_P2E8": 16.9420},
    {"Stage": "DS-B point", "Metric": "Global RMSE", "Paper1_or_P2E1": 23.9990, "Paper2_P2E8": 17.1300},
    {"Stage": "DS-A UQ", "Metric": "Balanced global coverage (%)", "Paper1_or_P2E1": 90.2100, "Paper2_P2E8": 85.0540},
    {"Stage": "DS-A UQ", "Metric": "Balanced mid-life coverage (%)", "Paper1_or_P2E1": 70.8800, "Paper2_P2E8": 81.5580},
    {"Stage": "DS-B UQ", "Metric": "Balanced global coverage (%)", "Paper1_or_P2E1": 90.6300, "Paper2_P2E8": 89.2050},
    {"Stage": "DS-B UQ", "Metric": "Balanced mid-life coverage (%)", "Paper1_or_P2E1": 75.1600, "Paper2_P2E8": 86.3490},
])

display(final_paper2_summary)

FINAL_SUMMARY_PATH = OUTPUT_DIR / "Paper2_final_headline_summary.csv"
final_paper2_summary.to_csv(FINAL_SUMMARY_PATH, index=False)
print("Saved:", FINAL_SUMMARY_PATH)


,Stage,Metric,Paper1_or_P2E1,Paper2_P2E8
0,IID matched,Global RMSE,19.1163,17.4316
1,IID matched,Mid-life RMSE,27.7041,23.7884
2,DS-A point,Global RMSE,24.8690,16.9420
3,DS-B point,Global RMSE,23.9990,17.1300
4,DS-A UQ,Balanced global coverage (%),90.2100,85.0540
5,DS-A UQ,Balanced mid-life coverage (%),70.8800,81.5580
6,DS-B UQ,Balanced global coverage (%),90.6300,89.2050
7,DS-B UQ,Balanced mid-life coverage (%),75.1600,86.3490


Saved: /content/drive/MyDrive/FD004_Paper2_Results/Paper2_final_headline_summary.csv


# 20. Provenance / source notebooks and frozen outputs

This Paper 2 notebook is based on and should remain auditable against the supplied Paper 1/Paper 2 research package, including:

- `02_FD004_Research_Baseline (5).ipynb`
- `03_FD004_DS_A_Heldout_Regime (1).ipynb`
- `04_FD004_DS_B_Restricted_Operating_Envelope (1).ipynb`
- `05_FD004_Paper1_Master_Comparison_and_Freeze (1).ipynb`
- `FD004_Paper2_Shift_V1_DS_A_P2E8_60cycle.ipynb`
- `FD004_Paper2_Shift_V2_DS_B_P2E8_60cycle.ipynb`
- `FD004_Paper2_UQ_V1_DS_A_P2E8_60cycle.ipynb`
- `FD004_Paper2_UQ_V1_DS_B_P2E8_60cycle.ipynb`
- `FD004_Final_Consolidated_Results_Report.docx` where applicable

Key frozen Paper 2 outputs include the P2-E8 checkpoint(s), `DS_A_P2E8_predictions.csv`, `DS_B_P2E8_predictions.csv`, `Shift_V3_persistent_engine_metrics.csv`, and the UQ-V1 DS-A/DS-B summary and engine-metric CSVs.

Before manuscript submission, retain the frozen Paper 1 artifacts alongside the Paper 2 outputs so every comparison remains traceable and auditable.
